# GVSTMR — Standalone Supplementary Figures S1–S11
## Fresh Google Colab runtime version

This notebook is **standalone**. You do **not** need to run the Main Figures notebook first.

### Upload these 3 required ZIP files when Cell 1 asks for them

1. `GVSTMR_FINAL_2015_2024.zip`
2. `GVSTMR_SIMULATION_CONUS_10T_V2.zip`
3. `GVSTMR_ANNUAL_GGPR_GEOSHAP_UNCERTAINTY_2015_2024.zip`

### What the bootstrap does

It reproduces only the analytical objects required by the Supplementary Material:

- loads the real 2015–2024 panel and CONUS hex grid;
- loads the simulation T1–T10 panel and grid;
- recomputes the **same direct GVSTMR operator used in the final manuscript**;
- reconstructs the saved MGWR, GTWR and GGPR model–observation GVSTMR objects;
- **does not redraw the main-manuscript figures**;
- **does not refit MGWR or GTWR**;
- GGPR uses the saved calibration and selected hyperparameters when the simulation time-resolved prediction table must be reconstructed.

Then Figures S1–S11 are generated.

> Important: keep the three ZIP filenames unchanged. The loader also searches by content, so minor internal-folder differences are tolerated.


In [ ]:

# ==========================================================================================
# CELL 1 — SETUP, UPLOAD THREE ZIP FILES, EXTRACT REQUIRED DATA
# ==========================================================================================

import sys
import subprocess
import importlib.util
import os
import gc
import json
import shutil
import zipfile
import warnings
import time
import math

from pathlib import Path


def ensure_package(package, module=None):
    module = module or package
    if importlib.util.find_spec(module) is None:
        print("Installing:", package)
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])


for pkg, mod in [
    ("geopandas", "geopandas"),
    ("pyogrio", "pyogrio"),
    ("pyarrow", "pyarrow"),
    ("scipy", "scipy"),
    ("scikit-learn", "sklearn"),
]:
    ensure_package(pkg, mod)


warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib
import matplotlib.pyplot as plt

from matplotlib.colors import (
    Normalize,
    TwoSlopeNorm,
    ListedColormap
)
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle, Ellipse
from matplotlib.gridspec import GridSpec

from scipy.spatial import cKDTree
from scipy.stats import spearmanr, chi2

from sklearn.preprocessing import StandardScaler


# ==========================================================================================
# GLOBAL SETTINGS
# ==========================================================================================

RANDOM_STATE = 123
rng = np.random.default_rng(RANDOM_STATE)

YEARS = list(range(2015, 2025))
SIM_TIMES = list(range(1, 11))

MAIN_YEARS = [2015, 2018, 2021, 2024]
MAIN_SIM_TIMES = [1, 4, 7, 10]

K_SPATIAL = 30
TEMPORAL_BANDWIDTH = 2.0

REAL_X = "NDVI"
REAL_Y = "LST_MEAN_C"

SELECTED_MGWR_FEATURES = [
    "TEMP_C",
    "NDVI",
    "RH",
    "SOIL_MM",
]

SELECTED_GTWR_FEATURES = [
    "TEMP_C",
    "NDVI",
]

FEATURES = [
    "TEMP_C",
    "SW_RAD_WM2",
    "RH",
    "WIND_MS",
    "PRECIP_MM",
    "NDVI",
    "SOIL_MM",
    "ELEV_M",
]


# ==========================================================================================
# OUTPUT
# ==========================================================================================

ROOT = Path("/content/GVSTMR_CARTOGRAPHY_MAIN_FIGURES")
FIG_ROOT = ROOT / "FIGURES"
PNG_ROOT = FIG_ROOT / "PNG_600DPI"
PDF_ROOT = FIG_ROOT / "PDF"
SVG_ROOT = FIG_ROOT / "SVG"
TIF_ROOT = FIG_ROOT / "TIFF_600DPI"
DATA_ROOT = ROOT / "FIGURE_DATA"
META_ROOT = ROOT / "METADATA"

ZIP_OUT = Path("/content/GVSTMR_CARTOGRAPHY_MAIN_FIGURES.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)

if ZIP_OUT.exists():
    ZIP_OUT.unlink()

for p in [
    PNG_ROOT,
    PDF_ROOT,
    SVG_ROOT,
    TIF_ROOT,
    DATA_ROOT,
    META_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)


# ==========================================================================================
# PUBLICATION STYLE
# ==========================================================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 9.0,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.2,
    "figure.titlesize": 11.0,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


# ==========================================================================================
# CORE GVSTMR PALETTE — SAME VISUAL GRAMMAR AS THE CONCEPT FIGURE
# ==========================================================================================

GV_PALETTE = {
    1: "#E5E4E9",
    2: "#B9D9E6",
    3: "#54B8D0",

    4: "#DB95CB",
    5: "#9E9FCB",
    6: "#4582BB",

    7: "#BE3F98",
    8: "#74529F",
    9: "#243A83",
}

# Separate uncertainty × effect palette from the annual analysis.
UE_FALLBACK_PALETTE = {
    1: "#f2f2f2",
    2: "#c7e9c0",
    3: "#41ab5d",
    4: "#d9d9d9",
    5: "#bcbddc",
    6: "#807dba",
    7: "#fdd0a2",
    8: "#fc8d59",
    9: "#d7301f",
}

TIME_COLORS = {
    1: "#762A83",
    4: "#C51B7D",
    7: "#4DAC26",
    10: "#276419",

    2015: "#762A83",
    2018: "#C51B7D",
    2021: "#4DAC26",
    2024: "#276419",
}


# ==========================================================================================
# LOCATE / UPLOAD ZIPS
# ==========================================================================================

def locate_zips():
    files = list(Path("/content").glob("*.zip"))

    real = [
        p for p in files
        if "GVSTMR_FINAL_2015_2024" in p.name
        and "ANNUAL" not in p.name
    ]

    sim = [
        p for p in files
        if "SIMULATION_CONUS_10T_V2" in p.name
    ]

    annual = [
        p for p in files
        if "ANNUAL_GGPR_GEOSHAP_UNCERTAINTY_2015_2024" in p.name
    ]

    return (
        real[0] if real else None,
        sim[0] if sim else None,
        annual[0] if annual else None,
    )


REAL_ZIP, SIM_ZIP, ANNUAL_ZIP = locate_zips()

if (
    REAL_ZIP is None
    or SIM_ZIP is None
    or ANNUAL_ZIP is None
):

    try:
        from google.colab import files

        print("\nUpload the THREE required ZIP files:")
        print("1) GVSTMR_FINAL_2015_2024.zip")
        print("2) GVSTMR_SIMULATION_CONUS_10T_V2.zip")
        print("3) GVSTMR_ANNUAL_GGPR_GEOSHAP_UNCERTAINTY_2015_2024.zip")

        files.upload()

    except Exception:
        pass


REAL_ZIP, SIM_ZIP, ANNUAL_ZIP = locate_zips()

if REAL_ZIP is None:
    raise RuntimeError("Missing GVSTMR_FINAL_2015_2024.zip")

if SIM_ZIP is None:
    raise RuntimeError("Missing GVSTMR_SIMULATION_CONUS_10T_V2.zip")

if ANNUAL_ZIP is None:
    raise RuntimeError(
        "Missing GVSTMR_ANNUAL_GGPR_GEOSHAP_UNCERTAINTY_2015_2024.zip"
    )


print("=" * 105)
print("GVSTMR FINAL CARTOGRAPHY")
print("=" * 105)

print("\nReal-world ZIP :", REAL_ZIP)
print("Simulation ZIP :", SIM_ZIP)
print("Annual GGPR ZIP:", ANNUAL_ZIP)


# ==========================================================================================
# ROBUST ZIP MEMBER HELPERS
# ==========================================================================================

WORK = Path("/content/_GVSTMR_CARTO_WORK")

if WORK.exists():
    shutil.rmtree(WORK)

WORK.mkdir(parents=True)


def zip_names(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        return z.namelist()


def find_member(
    zip_path,
    suffix=None,
    contains_all=None,
    contains_any=None,
    exclude=None
):
    names = zip_names(zip_path)

    matches = names

    if suffix is not None:
        matches = [
            n for n in matches
            if n.endswith(suffix)
        ]

    if contains_all:
        matches = [
            n for n in matches
            if all(
                token.lower() in n.lower()
                for token in contains_all
            )
        ]

    if contains_any:
        matches = [
            n for n in matches
            if any(
                token.lower() in n.lower()
                for token in contains_any
            )
        ]

    if exclude:
        matches = [
            n for n in matches
            if not any(
                token.lower() in n.lower()
                for token in exclude
            )
        ]

    if not matches:
        raise FileNotFoundError(
            f"No ZIP member found in {zip_path.name} "
            f"with suffix={suffix}, contains_all={contains_all}"
        )

    # Prefer the shortest / most direct path.
    matches = sorted(
        matches,
        key=lambda x: (
            x.count("/"),
            len(x)
        )
    )

    return matches[0]


def extract_member(zip_path, member, tag=None):
    tag = tag or zip_path.stem

    target = (
        WORK
        /
        tag
        /
        member
    )

    if not target.exists():
        target.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extract(
                member,
                WORK / tag
            )

    return target


def extract_by_suffix(
    zip_path,
    suffix,
    tag
):
    member = find_member(
        zip_path,
        suffix=suffix
    )

    return extract_member(
        zip_path,
        member,
        tag=tag
    )


def extract_by_tokens(
    zip_path,
    contains_all,
    tag,
    suffix=None,
    exclude=None
):
    member = find_member(
        zip_path,
        suffix=suffix,
        contains_all=contains_all,
        exclude=exclude
    )

    return extract_member(
        zip_path,
        member,
        tag=tag
    )


# ==========================================================================================
# LOAD REAL-WORLD CORE DATA
# ==========================================================================================

REAL_PANEL_PATH = extract_by_suffix(
    REAL_ZIP,
    "PANEL_2015_2024.parquet",
    "real"
)

REAL_HEX_PATH = extract_by_suffix(
    REAL_ZIP,
    "HEX_GRID.parquet",
    "real"
)

PANEL = pd.read_parquet(
    REAL_PANEL_PATH
)

HEX = gpd.read_parquet(
    REAL_HEX_PATH
)

PANEL["HEX_ID"] = PANEL["HEX_ID"].astype(str)
PANEL["YEAR"] = pd.to_numeric(
    PANEL["YEAR"],
    errors="coerce"
).astype("Int64")

HEX["HEX_ID"] = HEX["HEX_ID"].astype(str)

if HEX.crs is None:
    HEX = HEX.set_crs("EPSG:5070")
elif HEX.crs.to_epsg() != 5070:
    HEX = HEX.to_crs(5070)


# ==========================================================================================
# LOAD REAL MGWR
# ==========================================================================================

REAL_MGWR_PATH = extract_by_suffix(
    REAL_ZIP,
    "MGWR_FULL_LOCAL_RESULTS.parquet",
    "real"
)

MGWR_REAL = pd.read_parquet(
    REAL_MGWR_PATH
)

MGWR_REAL["HEX_ID"] = MGWR_REAL["HEX_ID"].astype(str)


# Bandwidth file is allowed to vary in exact name.
try:
    MGWR_BW_PATH = extract_by_tokens(
        REAL_ZIP,
        contains_all=["MGWR", "BANDWIDTH"],
        suffix=".csv",
        tag="real"
    )

    MGWR_BW = pd.read_csv(
        MGWR_BW_PATH
    )

except Exception:
    # Exact fallback from the completed full-CONUS MGWR run.
    MGWR_BW = pd.DataFrame({
        "VARIABLE": [
            "Intercept",
            "TEMP_C",
            "SW_RAD_WM2",
            "RH",
            "WIND_MS",
            "PRECIP_MM",
            "NDVI",
            "SOIL_MM",
            "ELEV_M",
        ],
        "BANDWIDTH_N_NEIGHBORS": [
            26,
            26,
            26,
            26,
            448,
            2937,
            26,
            26,
            26,
        ],
    })


# ==========================================================================================
# LOAD REAL GTWR
# ==========================================================================================

REAL_GTWR_PATH = extract_by_suffix(
    REAL_ZIP,
    "GTWR_FULL_LOCAL_RESULTS.parquet",
    "real"
)

GTWR_REAL = pd.read_parquet(
    REAL_GTWR_PATH
)

GTWR_REAL["HEX_ID"] = GTWR_REAL["HEX_ID"].astype(str)
GTWR_REAL["YEAR"] = pd.to_numeric(
    GTWR_REAL["YEAR"],
    errors="coerce"
).astype("Int64")


# ==========================================================================================
# LOAD SIMULATION CORE
# ==========================================================================================

try:
    SIM_PANEL_PATH = extract_by_suffix(
        SIM_ZIP,
        "GVSTMR_ALL_T1_T10_ATTRIBUTES.parquet",
        "sim"
    )

except Exception:
    SIM_PANEL_PATH = extract_by_suffix(
        SIM_ZIP,
        "SIMULATION_PANEL_T1_T10.parquet",
        "sim"
    )

SIM_PANEL = pd.read_parquet(
    SIM_PANEL_PATH
)

SIM_PANEL["HEX_ID"] = SIM_PANEL["HEX_ID"].astype(str)
SIM_PANEL["TIME"] = pd.to_numeric(
    SIM_PANEL["TIME"],
    errors="coerce"
).astype(int)


SIM_HEX_PATH = extract_by_suffix(
    SIM_ZIP,
    "SIMULATION_HEX_GRID.parquet",
    "sim"
)

SIM_HEX = gpd.read_parquet(
    SIM_HEX_PATH
)

SIM_HEX["HEX_ID"] = SIM_HEX["HEX_ID"].astype(str)

if SIM_HEX.crs is None:
    SIM_HEX = SIM_HEX.set_crs("EPSG:5070")
elif SIM_HEX.crs.to_epsg() != 5070:
    SIM_HEX = SIM_HEX.to_crs(5070)


# ==========================================================================================
# LOAD ANNUAL GGPR / GEOSHAP / UNCERTAINTY
# ==========================================================================================

ANNUAL_MAPS = {}

for year in YEARS:
    path = extract_by_suffix(
        ANNUAL_ZIP,
        f"ANNUAL_GGPR_GEOSHAP_UNCERTAINTY_{year}.parquet",
        "annual"
    )

    g = gpd.read_parquet(
        path
    )

    g["HEX_ID"] = g["HEX_ID"].astype(str)
    g["YEAR"] = pd.to_numeric(
        g["YEAR"],
        errors="coerce"
    ).astype(int)

    if g.crs is None:
        g = g.set_crs("EPSG:5070")
    elif g.crs.to_epsg() != 5070:
        g = g.to_crs(5070)

    ANNUAL_MAPS[year] = g


TEMP_COUPLING_PATH = extract_by_suffix(
    ANNUAL_ZIP,
    "TEMPORAL_UNCERTAINTY_GEOSHAP_COUPLING_MAP.parquet",
    "annual"
)

TEMP_COUPLING_MAP = gpd.read_parquet(
    TEMP_COUPLING_PATH
)

TEMP_COUPLING_MAP["HEX_ID"] = (
    TEMP_COUPLING_MAP["HEX_ID"].astype(str)
)

if TEMP_COUPLING_MAP.crs is None:
    TEMP_COUPLING_MAP = TEMP_COUPLING_MAP.set_crs("EPSG:5070")
elif TEMP_COUPLING_MAP.crs.to_epsg() != 5070:
    TEMP_COUPLING_MAP = TEMP_COUPLING_MAP.to_crs(5070)


TRAJECTORY_PATH = extract_by_suffix(
    ANNUAL_ZIP,
    "ANNUAL_UNCERTAINTY_EFFECT_TRAJECTORY.csv",
    "annual"
)

TRAJECTORY = pd.read_csv(
    TRAJECTORY_PATH
)


SPATIAL_COUPLING_PATH = extract_by_suffix(
    ANNUAL_ZIP,
    "YEARLY_SPATIAL_UNCERTAINTY_GEOSHAP_COUPLING.csv",
    "annual"
)

SPATIAL_COUPLING = pd.read_csv(
    SPATIAL_COUPLING_PATH
)


ANNUAL_METRICS_PATH = extract_by_suffix(
    ANNUAL_ZIP,
    "ANNUAL_GGPR_METRICS_2015_2024.csv",
    "annual"
)

ANNUAL_METRICS = pd.read_csv(
    ANNUAL_METRICS_PATH
)


try:
    UE_PALETTE_PATH = extract_by_suffix(
        ANNUAL_ZIP,
        "UNCERTAINTY_EFFECT_3X3_PALETTE.csv",
        "annual"
    )

    UE_TABLE = pd.read_csv(
        UE_PALETTE_PATH
    )

    UE_PALETTE = dict(
        zip(
            UE_TABLE["CELL"].astype(int),
            UE_TABLE["COLOR"]
        )
    )

except Exception:
    UE_PALETTE = UE_FALLBACK_PALETTE.copy()


# ==========================================================================================
# DATA SANITY
# ==========================================================================================

print("\nData loaded:")
print("PANEL             :", PANEL.shape)
print("HEX               :", HEX.shape)
print("MGWR              :", MGWR_REAL.shape)
print("GTWR              :", GTWR_REAL.shape)
print("SIM PANEL         :", SIM_PANEL.shape)
print("SIM HEX           :", SIM_HEX.shape)
print("Annual map 2015   :", ANNUAL_MAPS[2015].shape)
print("Temporal coupling :", TEMP_COUPLING_MAP.shape)

print("\n✅ Input data ready.")


In [ ]:

# ==========================================================================================
# CELL 2 — RECOMPUTE CORE DIRECT GVSTMR FOR SIMULATION + REAL-WORLD
#
# SAME METHOD FOR BOTH CASES:
#
# Spatial:
#   Pearson across focal hex + K nearest spatial neighbours
#
# Temporal:
#   Gaussian-weighted Pearson across all 10 time points,
#   centered on each target time, bandwidth h=2.
#
# ==========================================================================================

print("=" * 105)
print("DIRECT GVSTMR — SAME OPERATOR FOR SIMULATION AND REAL-WORLD")
print("=" * 105)

t0 = time.perf_counter()


# ==========================================================================================
# HELPERS
# ==========================================================================================

def ensure_xy_from_geometry(
    gdf
):
    g = gdf.copy()

    if (
        "X_KM" not in g.columns
        or
        "Y_KM" not in g.columns
    ):
        cent = g.geometry.centroid

        g["X_KM"] = (
            cent.x
            /
            1000.0
        )

        g["Y_KM"] = (
            cent.y
            /
            1000.0
        )

    return g


HEX = ensure_xy_from_geometry(
    HEX
)

SIM_HEX = ensure_xy_from_geometry(
    SIM_HEX
)


def rowwise_corr(
    A,
    B
):
    A = np.asarray(
        A,
        dtype=float
    )

    B = np.asarray(
        B,
        dtype=float
    )

    A0 = (
        A
        -
        A.mean(
            axis=1,
            keepdims=True
        )
    )

    B0 = (
        B
        -
        B.mean(
            axis=1,
            keepdims=True
        )
    )

    num = np.sum(
        A0
        *
        B0,
        axis=1
    )

    den = np.sqrt(
        np.sum(
            A0 ** 2,
            axis=1
        )
        *
        np.sum(
            B0 ** 2,
            axis=1
        )
    )

    return np.divide(
        num,
        den,
        out=np.zeros_like(
            num,
            dtype=float
        ),
        where=den > 1e-12
    )


def weighted_temporal_corr(
    X_mat,
    Y_mat,
    times,
    bandwidth=2.0
):
    """
    X_mat, Y_mat:
        [time, hex]

    Returns:
        [time, hex]
    """

    X_mat = np.asarray(
        X_mat,
        dtype=float
    )

    Y_mat = np.asarray(
        Y_mat,
        dtype=float
    )

    t = np.asarray(
        times,
        dtype=float
    )

    out = np.empty_like(
        X_mat,
        dtype=float
    )

    for c in range(
        len(
            t
        )
    ):
        w = np.exp(
            -0.5
            *
            (
                (
                    t
                    -
                    t[c]
                )
                /
                float(
                    bandwidth
                )
            ) ** 2
        )

        w = (
            w
            /
            w.sum()
        )

        mx = np.sum(
            w[
                :,
                None
            ]
            *
            X_mat,
            axis=0
        )

        my = np.sum(
            w[
                :,
                None
            ]
            *
            Y_mat,
            axis=0
        )

        dx = (
            X_mat
            -
            mx[
                None,
                :
            ]
        )

        dy = (
            Y_mat
            -
            my[
                None,
                :
            ]
        )

        cov = np.sum(
            w[
                :,
                None
            ]
            *
            dx
            *
            dy,
            axis=0
        )

        vx = np.sum(
            w[
                :,
                None
            ]
            *
            dx ** 2,
            axis=0
        )

        vy = np.sum(
            w[
                :,
                None
            ]
            *
            dy ** 2,
            axis=0
        )

        den = np.sqrt(
            vx
            *
            vy
        )

        out[
            c
        ] = np.divide(
            cov,
            den,
            out=np.zeros_like(
                cov
            ),
            where=den > 1e-12
        )

    return np.clip(
        out,
        -1,
        1
    )


def classify3(
    values,
    low_cut,
    high_cut
):
    return np.where(
        values <= low_cut,
        1,
        np.where(
            values <= high_cut,
            2,
            3
        )
    ).astype(
        np.int8
    )


def build_direct_gvstmr(
    panel,
    geometry,
    x_col,
    y_col,
    time_col,
    times,
    k_spatial=30,
    temporal_bandwidth=2.0,
    extra_cols=None,
):
    """
    Returns:
        long GeoDataFrame,
        thresholds dict
    """

    extra_cols = extra_cols or []

    P = panel.copy()
    P["HEX_ID"] = P["HEX_ID"].astype(str)

    G = ensure_xy_from_geometry(
        geometry
    ).copy()

    G["HEX_ID"] = G["HEX_ID"].astype(str)

    # One static coordinate record per hex.
    static = (
        G[
            [
                "HEX_ID",
                "X_KM",
                "Y_KM",
                "geometry"
            ]
        ]
        .drop_duplicates(
            "HEX_ID"
        )
        .copy()
    )

    P = P.loc[
        P[
            time_col
        ].isin(
            times
        )
    ].copy()

    # Require complete x/y time series for all ten times.
    complete = (
        P
        .groupby(
            "HEX_ID"
        )
        .agg(
            N_TIME=(
                time_col,
                "nunique"
            ),
            NX=(
                x_col,
                lambda s: int(
                    s.notna().sum()
                )
            ),
            NY=(
                y_col,
                lambda s: int(
                    s.notna().sum()
                )
            ),
        )
    )

    ids = complete.index[
        (
            complete["N_TIME"].eq(
                len(
                    times
                )
            )
            &
            complete["NX"].eq(
                len(
                    times
                )
            )
            &
            complete["NY"].eq(
                len(
                    times
                )
            )
        )
    ].astype(str).tolist()

    # Preserve geometry order for KNN and pivots.
    order = (
        static.loc[
            static[
                "HEX_ID"
            ].isin(
                ids
            ),
            "HEX_ID"
        ]
        .astype(str)
        .tolist()
    )

    static = (
        static
        .set_index(
            "HEX_ID"
        )
        .loc[
            order
        ]
        .reset_index()
    )

    pp = P.loc[
        P[
            "HEX_ID"
        ].isin(
            order
        )
    ].copy()

    X = (
        pp
        .pivot(
            index=time_col,
            columns="HEX_ID",
            values=x_col
        )
        .loc[
            times,
            order
        ]
        .to_numpy(
            float
        )
    )

    Y = (
        pp
        .pivot(
            index=time_col,
            columns="HEX_ID",
            values=y_col
        )
        .loc[
            times,
            order
        ]
        .to_numpy(
            float
        )
    )

    coords = static[
        [
            "X_KM",
            "Y_KM"
        ]
    ].to_numpy(
        float
    )

    tree = cKDTree(
        coords
    )

    _, nbr = tree.query(
        coords,
        k=min(
            k_spatial + 1,
            len(
                coords
            )
        )
    )

    rho_s = np.empty_like(
        X,
        dtype=float
    )

    for ti in range(
        len(
            times
        )
    ):
        rho_s[
            ti
        ] = rowwise_corr(
            X[
                ti
            ][
                nbr
            ],
            Y[
                ti
            ][
                nbr
            ]
        )

    rho_t = weighted_temporal_corr(
        X,
        Y,
        times=np.arange(
            len(
                times
            ),
            dtype=float
        ),
        bandwidth=temporal_bandwidth
    )

    s_low, s_high = np.quantile(
        rho_s.ravel(),
        [
            1 / 3,
            2 / 3
        ]
    )

    t_low, t_high = np.quantile(
        rho_t.ravel(),
        [
            1 / 3,
            2 / 3
        ]
    )

    s_class = classify3(
        rho_s,
        s_low,
        s_high
    )

    t_class = classify3(
        rho_t,
        t_low,
        t_high
    )

    cell = (
        (
            s_class
            -
            1
        )
        *
        3
        +
        t_class
    ).astype(
        np.int8
    )

    rows = []

    for ti, tv in enumerate(
        times
    ):
        D = pd.DataFrame({
            "HEX_ID":
                order,

            time_col:
                tv,

            x_col:
                X[
                    ti
                ],

            y_col:
                Y[
                    ti
                ],

            "RHO_S":
                rho_s[
                    ti
                ],

            "RHO_T":
                rho_t[
                    ti
                ],

            "SPATIAL_CLASS":
                s_class[
                    ti
                ],

            "TEMPORAL_CLASS":
                t_class[
                    ti
                ],

            "GV_CELL":
                cell[
                    ti
                ],
        })

        for col in extra_cols:
            if col in pp.columns:
                vals = (
                    pp.loc[
                        pp[
                            time_col
                        ].eq(
                            tv
                        )
                    ]
                    .set_index(
                        "HEX_ID"
                    )
                    .reindex(
                        order
                    )[
                        col
                    ]
                    .to_numpy()
                )

                D[
                    col
                ] = vals

        D = static.merge(
            D,
            on="HEX_ID",
            how="inner",
            validate="1:1"
        )

        D = gpd.GeoDataFrame(
            D,
            geometry="geometry",
            crs=G.crs
        )

        rows.append(
            D
        )

    OUT = pd.concat(
        rows,
        ignore_index=True
    )

    OUT = gpd.GeoDataFrame(
        OUT,
        geometry="geometry",
        crs=G.crs
    )

    thresholds = {
        "SPATIAL_LOW":
            float(
                s_low
            ),

        "SPATIAL_HIGH":
            float(
                s_high
            ),

        "TEMPORAL_LOW":
            float(
                t_low
            ),

        "TEMPORAL_HIGH":
            float(
                t_high
            ),

        "K_SPATIAL":
            int(
                k_spatial
            ),

        "TEMPORAL_BANDWIDTH":
            float(
                temporal_bandwidth
            ),
    }

    return (
        OUT,
        thresholds
    )


# ==========================================================================================
# SIMULATION
# ==========================================================================================

sim_extra = [
    c
    for c in [
        "TRUE_BETA_X",
        "Y_SIGNAL"
    ]
    if c in SIM_PANEL.columns
]

SIM_DIRECT, SIM_THRESHOLDS = build_direct_gvstmr(
    panel=SIM_PANEL,
    geometry=SIM_HEX,
    x_col="X",
    y_col="Y",
    time_col="TIME",
    times=SIM_TIMES,
    k_spatial=K_SPATIAL,
    temporal_bandwidth=TEMPORAL_BANDWIDTH,
    extra_cols=sim_extra,
)


# ==========================================================================================
# REAL-WORLD: NDVI ↔ LST
# ==========================================================================================

REAL_DIRECT, REAL_THRESHOLDS = build_direct_gvstmr(
    panel=PANEL,
    geometry=HEX,
    x_col=REAL_X,
    y_col=REAL_Y,
    time_col="YEAR",
    times=YEARS,
    k_spatial=K_SPATIAL,
    temporal_bandwidth=TEMPORAL_BANDWIDTH,
    extra_cols=[
        "TEMP_C"
    ],
)


# ==========================================================================================
# SAVE
# ==========================================================================================

SIM_DIRECT.to_parquet(
    DATA_ROOT /
    "DIRECT_GVSTMR_SIMULATION_T1_T10.parquet",
    index=False
)

REAL_DIRECT.to_parquet(
    DATA_ROOT /
    "DIRECT_GVSTMR_REAL_NDVI_LST_2015_2024.parquet",
    index=False
)


with open(
    META_ROOT /
    "DIRECT_GVSTMR_THRESHOLDS.json",
    "w"
) as f:
    json.dump(
        {
            "simulation":
                SIM_THRESHOLDS,

            "real_world_NDVI_LST":
                REAL_THRESHOLDS,
        },
        f,
        indent=2
    )


# ==========================================================================================
# SIMULATION KNOWN-EFFECT RECOVERY
# ==========================================================================================

if "TRUE_BETA_X" in SIM_DIRECT.columns:
    ok = (
        SIM_DIRECT[
            "TRUE_BETA_X"
        ].notna()
    )

    SIM_RECOVERY = pd.DataFrame([{
        "SPATIAL_RHO_vs_TRUE_BETA_SPEARMAN":
            float(
                spearmanr(
                    SIM_DIRECT.loc[
                        ok,
                        "RHO_S"
                    ],
                    SIM_DIRECT.loc[
                        ok,
                        "TRUE_BETA_X"
                    ]
                ).statistic
            ),

        "TEMPORAL_RHO_vs_TRUE_BETA_SPEARMAN":
            float(
                spearmanr(
                    SIM_DIRECT.loc[
                        ok,
                        "RHO_T"
                    ],
                    SIM_DIRECT.loc[
                        ok,
                        "TRUE_BETA_X"
                    ]
                ).statistic
            ),

        "N":
            int(
                ok.sum()
            ),
    }])

else:
    SIM_RECOVERY = pd.DataFrame([{
        "SPATIAL_RHO_vs_TRUE_BETA_SPEARMAN":
            np.nan,

        "TEMPORAL_RHO_vs_TRUE_BETA_SPEARMAN":
            np.nan,

        "N":
            len(
                SIM_DIRECT
            ),
    }])


SIM_RECOVERY.to_csv(
    DATA_ROOT /
    "SIMULATION_DIRECT_GVSTMR_RECOVERY.csv",
    index=False
)


print("\nSimulation thresholds:")
print(SIM_THRESHOLDS)

print("\nReal-world thresholds:")
print(REAL_THRESHOLDS)

print("\nSimulation direct-score recovery:")
display(
    SIM_RECOVERY
)

print(
    f"\nDirect GVSTMR complete in {(time.perf_counter()-t0)/60:.1f} min"
)


In [ ]:
# ==========================================================================================
# CELL 0 — COMMON GVSTMR ENGINE
# Run ONCE before MGWR / GTWR / GGPR cells
#
# OUTPUT: /content/GVSTMR_MODELWISE_GVSTMR
# NO GOOGLE DRIVE WRITE
# ==========================================================================================

from pathlib import Path
import zipfile
import shutil
import warnings
import re
import json
import gc

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

warnings.filterwarnings("ignore")

# ==========================================================================================
# SETTINGS
# ==========================================================================================

CONTENT = Path("/content")

WORK = CONTENT / "GVSTMR_MODELWISE_GVSTMR"
EXTRACT = WORK / "_extracted"
OUTPUT = WORK / "outputs"

EXTRACT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2015, 2025))
SIM_TIMES = list(range(1, 11))

K_SPATIAL = 30

# Conceptual figure:
# t-2, t-1, t, t+1, t+2
TEMP_HALF_WINDOW = 2

# Same 3×3 GVSTMR color grammar everywhere
# row = spatial Low/Medium/High
# col = temporal Low/Medium/High
GV_PALETTE = {
    1: "#E5E4E9",  # Low S / Low T
    2: "#B9D9E6",  # Low S / Med T
    3: "#54B8D0",  # Low S / High T

    4: "#DB95CB",  # Med S / Low T
    5: "#9E9FCB",  # Med S / Med T
    6: "#4582BB",  # Med S / High T

    7: "#BE3F98",  # High S / Low T
    8: "#74529F",  # High S / Med T
    9: "#243A83",  # High S / High T
}

CLASS_NAME = {
    1: "Low",
    2: "Medium",
    3: "High"
}

print("=" * 110)
print("GVSTMR — MODEL-vs-OBSERVATION ENGINE")
print("=" * 110)
print("Spatial neighborhood :", K_SPATIAL, "nearest neighbors + focal hex")
print("Temporal window      :", f"t±{TEMP_HALF_WINDOW}")
print("Output               :", OUTPUT)
print("Google Drive write   : NO")


# ==========================================================================================
# 1. EXTRACT GVSTMR ZIP FILES LOCALLY
# ==========================================================================================

zip_files = sorted(
    p for p in CONTENT.glob("GVSTMR*.zip")
    if p.is_file()
)

print("\nZIP files found:")
for p in zip_files:
    print(" -", p.name)

for zp in zip_files:

    dest = EXTRACT / zp.stem
    marker = dest / "_EXTRACTION_COMPLETE.txt"

    if marker.exists():
        continue

    dest.mkdir(parents=True, exist_ok=True)

    print("\nExtracting:", zp.name)

    with zipfile.ZipFile(zp, "r") as z:
        z.extractall(dest)

    marker.write_text(
        f"Extracted from {zp.name}",
        encoding="utf-8"
    )


# ==========================================================================================
# 2. FIND TABLES — NEVER SEARCH GOOGLE DRIVE
# ==========================================================================================

def refresh_table_files():

    files = []

    for suffix in ("*.parquet", "*.csv"):

        for p in CONTENT.rglob(suffix):

            s = str(p)

            if s.startswith("/content/drive/"):
                continue

            # Do not use our own generated outputs as model inputs
            if str(OUTPUT) in s:
                continue

            files.append(p)

    return sorted(set(files))


TABLE_FILES = refresh_table_files()

print("\nLocal input tables indexed:", f"{len(TABLE_FILES):,}")


def read_table(path):

    path = Path(path)

    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    raise ValueError(path)


def norm_name(x):

    return re.sub(
        r"[^A-Z0-9]+",
        "",
        str(x).upper()
    )


def choose_column(df, candidates, required=True):

    # exact first
    for c in candidates:
        if c in df.columns:
            return c

    normalized = {
        norm_name(c): c
        for c in df.columns
    }

    for c in candidates:
        nc = norm_name(c)

        if nc in normalized:
            return normalized[nc]

    if required:
        raise KeyError(
            f"Could not find any of {candidates}\n"
            f"Available columns:\n{list(df.columns)}"
        )

    return None


def locate_exact_table(
    basenames,
    required_cols=None,
    prefer_terms=None,
    exclude_terms=None
):

    required_cols = required_cols or []
    prefer_terms = [
        str(x).upper()
        for x in (prefer_terms or [])
    ]
    exclude_terms = [
        str(x).upper()
        for x in (exclude_terms or [])
    ]

    wanted = {
        str(x).lower()
        for x in basenames
    }

    candidates = [
        p for p in TABLE_FILES
        if p.name.lower() in wanted
    ]

    def score(p):

        s = str(p).upper()

        return sum(
            5 for x in prefer_terms
            if x in s
        )

    candidates = sorted(
        candidates,
        key=score,
        reverse=True
    )

    for p in candidates:

        ps = str(p).upper()

        if any(x in ps for x in exclude_terms):
            continue

        try:
            d = read_table(p)

        except Exception:
            continue

        if all(c in d.columns for c in required_cols):

            print("Using:", p)
            return d, p

    return None, None


def locate_table_by_columns(
    required_cols,
    path_terms=None,
    exclude_terms=None,
    required_any_tokens=None
):

    path_terms = [
        str(x).upper()
        for x in (path_terms or [])
    ]

    exclude_terms = [
        str(x).upper()
        for x in (exclude_terms or [])
    ]

    required_any_tokens = [
        str(x).upper()
        for x in (required_any_tokens or [])
    ]

    def score(p):

        s = str(p).upper()

        q = sum(
            4 for x in path_terms
            if x in s
        )

        q += sum(
            1 for x in required_any_tokens
            if x in p.name.upper()
        )

        return q

    candidates = sorted(
        TABLE_FILES,
        key=score,
        reverse=True
    )

    for p in candidates:

        ps = str(p).upper()

        if any(x in ps for x in exclude_terms):
            continue

        # Avoid opening completely unrelated files first
        if path_terms and not any(
            x in ps
            for x in path_terms
        ):
            continue

        try:
            d = read_table(p)

        except Exception:
            continue

        if not all(
            c in d.columns
            for c in required_cols
        ):
            continue

        if required_any_tokens:

            all_text = " ".join(
                str(c).upper()
                for c in d.columns
            )

            if not any(
                x in all_text
                for x in required_any_tokens
            ):
                continue

        print("Using:", p)
        return d, p

    return None, None


# ==========================================================================================
# 3. REAL PANEL
# ==========================================================================================

def get_real_panel():

    # First use existing runtime PANEL if available.
    if "PANEL" in globals():

        p = globals()["PANEL"]

        if isinstance(p, pd.DataFrame):

            if {
                "HEX_ID",
                "YEAR",
                "LST_MEAN_C"
            }.issubset(p.columns):

                print("Using runtime object: PANEL")
                return p.copy()

    names = [
        "CONUS_HEX_YEAR_LST_RESULTS.parquet",
        "CONUS_HEX_YEAR_LST_PANEL.parquet",
        "CONUS_HEX_YEAR_LST_RESULTS.csv",
        "CONUS_HEX_YEAR_LST_PANEL.csv",
    ]

    d, p = locate_exact_table(
        names,
        required_cols=[
            "HEX_ID",
            "YEAR",
            "LST_MEAN_C"
        ]
    )

    if d is not None:
        return d

    d, p = locate_table_by_columns(
        required_cols=[
            "HEX_ID",
            "YEAR",
            "LST_MEAN_C"
        ],
        path_terms=[
            "2015_2024",
            "MODEL_INPUT",
            "HEX_YEAR",
            "PANEL"
        ]
    )

    if d is None:

        raise RuntimeError(
            "\nCould not locate real 2015–2024 panel.\n"
            "Expected a table containing:\n"
            "HEX_ID, YEAR, LST_MEAN_C\n"
        )

    return d


# ==========================================================================================
# 4. SIMULATION PANEL
# ==========================================================================================

def get_sim_panel():

    if "SIM_PANEL" in globals():

        s = globals()["SIM_PANEL"]

        if isinstance(s, pd.DataFrame):

            if {
                "HEX_ID",
                "TIME",
                "X",
                "Y"
            }.issubset(s.columns):

                print("Using runtime object: SIM_PANEL")
                return s.copy()

    d, p = locate_exact_table(
        [
            "SIMULATION_PANEL_T1_T10.parquet",
            "GVSTMR_ALL_T1_T10_ATTRIBUTES.parquet"
        ],
        required_cols=[
            "HEX_ID",
            "TIME",
            "X",
            "Y"
        ],
        prefer_terms=[
            "SIMULATION",
            "V2"
        ]
    )

    if d is None:

        raise RuntimeError(
            "\nCould not locate Simulation T1–T10 panel.\n"
            "Expected SIMULATION_PANEL_T1_T10.parquet\n"
        )

    return d


# ==========================================================================================
# 5. ROW-WISE COVARIANCE + CORRELATION
#
# Gives the actual covariance-matrix quantities used in the conceptual figure:
#
# [ Var(Model)       Cov(Model,Obs) ]
# [ Cov(Model,Obs)   Var(Obs)       ]
#
# rho = Cov / sqrt(Var_model * Var_obs)
# ==========================================================================================

def row_covcorr(A, B, min_n=3):

    A = np.asarray(A, float)
    B = np.asarray(B, float)

    valid = (
        np.isfinite(A)
        &
        np.isfinite(B)
    )

    n = valid.sum(axis=1)

    Av = np.where(valid, A, 0.0)
    Bv = np.where(valid, B, 0.0)

    sumA = Av.sum(axis=1)
    sumB = Bv.sum(axis=1)

    meanA = np.divide(
        sumA,
        n,
        out=np.full(len(n), np.nan),
        where=n > 0
    )

    meanB = np.divide(
        sumB,
        n,
        out=np.full(len(n), np.nan),
        where=n > 0
    )

    da = np.where(
        valid,
        A - meanA[:, None],
        0.0
    )

    db = np.where(
        valid,
        B - meanB[:, None],
        0.0
    )

    den_n = np.maximum(
        n - 1,
        1
    )

    varA = (
        (da * da).sum(axis=1)
        /
        den_n
    )

    varB = (
        (db * db).sum(axis=1)
        /
        den_n
    )

    cov = (
        (da * db).sum(axis=1)
        /
        den_n
    )

    denom = np.sqrt(
        varA * varB
    )

    rho = np.divide(
        cov,
        denom,
        out=np.full(len(n), np.nan),
        where=denom > 1e-12
    )

    bad = (
        (n < min_n)
        |
        (~np.isfinite(rho))
    )

    varA[bad] = np.nan
    varB[bad] = np.nan
    cov[bad] = np.nan
    rho[bad] = np.nan

    rho = np.clip(
        rho,
        -1.0,
        1.0
    )

    return varA, varB, cov, rho, n


# ==========================================================================================
# 6. CLASSIFICATION
# ==========================================================================================

def classify_3(values, low_cut, high_cut):

    v = np.asarray(values, float)

    out = np.full(
        len(v),
        np.nan
    )

    ok = np.isfinite(v)

    out[
        ok & (v <= low_cut)
    ] = 1

    out[
        ok
        & (v > low_cut)
        & (v <= high_cut)
    ] = 2

    out[
        ok & (v > high_cut)
    ] = 3

    return out


# ==========================================================================================
# 7. MAIN GVSTMR MODEL-vs-OBSERVATION FUNCTION
# ==========================================================================================

def build_model_gvstmr(
    df,
    id_col,
    time_col,
    model_col,
    obs_col,
    x_col,
    y_col,
    case_name,
    model_name,
    output_dir,
    keep_cols=None,
    k_spatial=K_SPATIAL,
    half_window=TEMP_HALF_WINDOW
):

    print("\n" + "=" * 110)
    print(model_name, "|", case_name)
    print("=" * 110)

    keep_cols = keep_cols or []

    cols = [
        id_col,
        time_col,
        model_col,
        obs_col,
        x_col,
        y_col,
    ]

    cols += [
        c for c in keep_cols
        if c in df.columns
        and c not in cols
    ]

    d = df[cols].copy()

    d = d.rename(
        columns={
            id_col: "HEX_ID",
            time_col: "TIME",
            model_col: "MODEL_VALUE",
            obs_col: "OBS_VALUE",
            x_col: "X_KM",
            y_col: "Y_KM",
        }
    )

    d["HEX_ID"] = (
        d["HEX_ID"]
        .astype(str)
    )

    d["TIME"] = pd.to_numeric(
        d["TIME"],
        errors="coerce"
    )

    for c in [
        "MODEL_VALUE",
        "OBS_VALUE",
        "X_KM",
        "Y_KM"
    ]:

        d[c] = pd.to_numeric(
            d[c],
            errors="coerce"
        )

    d = d.dropna(
        subset=[
            "HEX_ID",
            "TIME",
            "X_KM",
            "Y_KM"
        ]
    )

    dup = d.duplicated(
        ["HEX_ID", "TIME"]
    )

    if dup.any():

        raise RuntimeError(
            f"{model_name} {case_name}: "
            f"{dup.sum():,} duplicate HEX_ID × TIME rows found."
        )

    times = sorted(
        d["TIME"]
        .dropna()
        .unique()
        .tolist()
    )

    coords = (
        d[
            [
                "HEX_ID",
                "X_KM",
                "Y_KM"
            ]
        ]
        .drop_duplicates("HEX_ID")
        .sort_values("HEX_ID")
        .reset_index(drop=True)
    )

    ids = coords[
        "HEX_ID"
    ].tolist()

    n_hex = len(ids)
    n_time = len(times)

    print("Hexes :", f"{n_hex:,}")
    print("Times :", times)

    # ------------------------------------------------------------------
    # Matrices: time × hex
    # ------------------------------------------------------------------

    MODEL = (
        d.pivot(
            index="TIME",
            columns="HEX_ID",
            values="MODEL_VALUE"
        )
        .reindex(
            index=times,
            columns=ids
        )
        .to_numpy(float)
    )

    OBS = (
        d.pivot(
            index="TIME",
            columns="HEX_ID",
            values="OBS_VALUE"
        )
        .reindex(
            index=times,
            columns=ids
        )
        .to_numpy(float)
    )

    # ------------------------------------------------------------------
    # Spatial neighbors
    # focal + k nearest
    # ------------------------------------------------------------------

    xy = coords[
        ["X_KM", "Y_KM"]
    ].to_numpy(float)

    tree = cKDTree(xy)

    k_query = min(
        int(k_spatial) + 1,
        n_hex
    )

    _, nbr = tree.query(
        xy,
        k=k_query
    )

    if nbr.ndim == 1:
        nbr = nbr[:, None]

    # ------------------------------------------------------------------
    # Output arrays
    # ------------------------------------------------------------------

    shape = (
        n_time,
        n_hex
    )

    VAR_MODEL_S = np.full(
        shape,
        np.nan
    )

    VAR_OBS_S = np.full(
        shape,
        np.nan
    )

    COV_S = np.full(
        shape,
        np.nan
    )

    RHO_S = np.full(
        shape,
        np.nan
    )

    N_S = np.zeros(
        shape,
        dtype=np.int16
    )

    VAR_MODEL_T = np.full(
        shape,
        np.nan
    )

    VAR_OBS_T = np.full(
        shape,
        np.nan
    )

    COV_T = np.full(
        shape,
        np.nan
    )

    RHO_T = np.full(
        shape,
        np.nan
    )

    N_T = np.zeros(
        shape,
        dtype=np.int16
    )

    # ------------------------------------------------------------------
    # A. SPATIAL covariance/correlation
    # ------------------------------------------------------------------

    for ti in range(n_time):

        A = MODEL[
            ti
        ][nbr]

        B = OBS[
            ti
        ][nbr]

        va, vb, cov, rho, nn = row_covcorr(
            A,
            B,
            min_n=3
        )

        VAR_MODEL_S[ti] = va
        VAR_OBS_S[ti] = vb
        COV_S[ti] = cov
        RHO_S[ti] = rho
        N_S[ti] = nn

    # ------------------------------------------------------------------
    # B. TEMPORAL covariance/correlation
    #
    # conceptual centered window:
    # t-2 ... t ... t+2
    #
    # At first/last years only available observations are used.
    # Minimum = 3 paired times.
    # ------------------------------------------------------------------

    for ti in range(n_time):

        lo = max(
            0,
            ti - half_window
        )

        hi = min(
            n_time,
            ti + half_window + 1
        )

        A = MODEL[
            lo:hi
        ].T

        B = OBS[
            lo:hi
        ].T

        va, vb, cov, rho, nn = row_covcorr(
            A,
            B,
            min_n=3
        )

        VAR_MODEL_T[ti] = va
        VAR_OBS_T[ti] = vb
        COV_T[ti] = cov
        RHO_T[ti] = rho
        N_T[ti] = nn

    # ------------------------------------------------------------------
    # C. Build long output
    # ------------------------------------------------------------------

    result = pd.MultiIndex.from_product(
        [
            times,
            ids
        ],
        names=[
            "TIME",
            "HEX_ID"
        ]
    ).to_frame(
        index=False
    )

    result[
        "MODEL_VALUE"
    ] = MODEL.reshape(-1)

    result[
        "OBS_VALUE"
    ] = OBS.reshape(-1)

    result[
        "VAR_MODEL_S"
    ] = VAR_MODEL_S.reshape(-1)

    result[
        "VAR_OBS_S"
    ] = VAR_OBS_S.reshape(-1)

    result[
        "COV_MODEL_OBS_S"
    ] = COV_S.reshape(-1)

    result[
        "RHO_S"
    ] = RHO_S.reshape(-1)

    result[
        "N_SPATIAL_PAIRS"
    ] = N_S.reshape(-1)

    result[
        "VAR_MODEL_T"
    ] = VAR_MODEL_T.reshape(-1)

    result[
        "VAR_OBS_T"
    ] = VAR_OBS_T.reshape(-1)

    result[
        "COV_MODEL_OBS_T"
    ] = COV_T.reshape(-1)

    result[
        "RHO_T"
    ] = RHO_T.reshape(-1)

    result[
        "N_TEMPORAL_PAIRS"
    ] = N_T.reshape(-1)

    result = result.merge(
        coords,
        on="HEX_ID",
        how="left",
        validate="m:1"
    )

    # ------------------------------------------------------------------
    # D. Fixed global tertiles pooled across ALL times in this case/model
    # ------------------------------------------------------------------

    valid_s = result[
        "RHO_S"
    ].to_numpy(float)

    valid_s = valid_s[
        np.isfinite(valid_s)
    ]

    valid_t = result[
        "RHO_T"
    ].to_numpy(float)

    valid_t = valid_t[
        np.isfinite(valid_t)
    ]

    if len(valid_s) == 0 or len(valid_t) == 0:

        raise RuntimeError(
            f"{model_name} {case_name}: "
            "No valid rho values were produced."
        )

    S_LOW, S_HIGH = np.quantile(
        valid_s,
        [
            1 / 3,
            2 / 3
        ]
    )

    T_LOW, T_HIGH = np.quantile(
        valid_t,
        [
            1 / 3,
            2 / 3
        ]
    )

    result[
        "SPATIAL_CLASS"
    ] = classify_3(
        result["RHO_S"],
        S_LOW,
        S_HIGH
    )

    result[
        "TEMPORAL_CLASS"
    ] = classify_3(
        result["RHO_T"],
        T_LOW,
        T_HIGH
    )

    result[
        "GV_CELL"
    ] = np.where(
        np.isfinite(
            result[
                "SPATIAL_CLASS"
            ]
        )
        &
        np.isfinite(
            result[
                "TEMPORAL_CLASS"
            ]
        ),
        (
            (
                result[
                    "SPATIAL_CLASS"
                ]
                -
                1
            )
            *
            3
            +
            result[
                "TEMPORAL_CLASS"
            ]
        ),
        np.nan
    )

    result[
        "GV_COLOR"
    ] = (
        result[
            "GV_CELL"
        ]
        .map(
            GV_PALETTE
        )
    )

    result[
        "SPATIAL_LEVEL"
    ] = (
        result[
            "SPATIAL_CLASS"
        ]
        .map(
            CLASS_NAME
        )
    )

    result[
        "TEMPORAL_LEVEL"
    ] = (
        result[
            "TEMPORAL_CLASS"
        ]
        .map(
            CLASS_NAME
        )
    )

    result[
        "MODEL"
    ] = model_name

    result[
        "CASE"
    ] = case_name

    # ------------------------------------------------------------------
    # Reattach requested extra columns
    # ------------------------------------------------------------------

    extra = [
        c for c in keep_cols
        if c in d.columns
        and c not in result.columns
    ]

    if extra:

        e = d[
            [
                "HEX_ID",
                "TIME",
                *extra
            ]
        ].copy()

        result = result.merge(
            e,
            on=[
                "HEX_ID",
                "TIME"
            ],
            how="left",
            validate="1:1"
        )

    # ------------------------------------------------------------------
    # E. Metrics — SUPPORTING diagnostics only
    # NEVER used to construct GVSTMR cells
    # ------------------------------------------------------------------

    yy = result[
        "OBS_VALUE"
    ].to_numpy(float)

    pp = result[
        "MODEL_VALUE"
    ].to_numpy(float)

    ok = (
        np.isfinite(yy)
        &
        np.isfinite(pp)
    )

    yy = yy[ok]
    pp = pp[ok]

    metrics = pd.DataFrame([{
        "MODEL":
            model_name,

        "CASE":
            case_name,

        "N":
            len(yy),

        "R2":
            r2_score(
                yy,
                pp
            )
            if len(yy) >= 3
            else np.nan,

        "RMSE":
            mean_squared_error(
                yy,
                pp
            ) ** 0.5
            if len(yy)
            else np.nan,

        "MAE":
            mean_absolute_error(
                yy,
                pp
            )
            if len(yy)
            else np.nan,

        "BIAS":
            float(
                np.mean(
                    pp - yy
                )
            )
            if len(yy)
            else np.nan,

        "PEARSON_R":
            float(
                np.corrcoef(
                    yy,
                    pp
                )[0, 1]
            )
            if len(yy) >= 3
            else np.nan,

        "MEAN_RHO_S":
            float(
                np.nanmean(
                    result[
                        "RHO_S"
                    ]
                )
            ),

        "MEDIAN_RHO_S":
            float(
                np.nanmedian(
                    result[
                        "RHO_S"
                    ]
                )
            ),

        "MEAN_RHO_T":
            float(
                np.nanmean(
                    result[
                        "RHO_T"
                    ]
                )
            ),

        "MEDIAN_RHO_T":
            float(
                np.nanmedian(
                    result[
                        "RHO_T"
                    ]
                )
            ),
    }])

    cuts = pd.DataFrame([{
        "MODEL":
            model_name,

        "CASE":
            case_name,

        "K_SPATIAL":
            k_spatial,

        "TEMP_HALF_WINDOW":
            half_window,

        "SPATIAL_LOW_CUT":
            S_LOW,

        "SPATIAL_HIGH_CUT":
            S_HIGH,

        "TEMPORAL_LOW_CUT":
            T_LOW,

        "TEMPORAL_HIGH_CUT":
            T_HIGH,
    }])

    # ------------------------------------------------------------------
    # SAVE
    # ------------------------------------------------------------------

    output_dir = Path(
        output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    prefix = (
        f"{model_name}_{case_name}"
        .upper()
        .replace(" ", "_")
        .replace("-", "_")
    )

    result.to_parquet(
        output_dir
        /
        f"{prefix}_GVSTMR_ALL_TIMES.parquet",
        index=False
    )

    cuts.to_csv(
        output_dir
        /
        f"{prefix}_GVSTMR_THRESHOLDS.csv",
        index=False
    )

    metrics.to_csv(
        output_dir
        /
        f"{prefix}_MODEL_METRICS.csv",
        index=False
    )

    for tt in times:

        q = result.loc[
            result[
                "TIME"
            ].eq(tt)
        ].copy()

        q.to_parquet(
            output_dir
            /
            f"{prefix}_GVSTMR_{int(tt)}.parquet",
            index=False
        )

    print("\nThresholds:")
    display(cuts)

    print("\nModel diagnostics:")
    display(metrics)

    print(
        "\nSaved:",
        output_dir
    )

    return result, cuts, metrics


print("\n✅ COMMON GVSTMR ENGINE READY")
print("NO FILE WRITTEN TO GOOGLE DRIVE.")

In [ ]:
# ==========================================================================================
# CELL 1 — MGWR → GVSTMR
#
# REAL       : 2015–2024
# SIMULATION : T1–T10
#
# NO MGWR REFIT
# NO GOOGLE DRIVE WRITE
# ==========================================================================================

print("=" * 110)
print("MGWR → MODEL-vs-OBSERVATION GVSTMR")
print("=" * 110)

MGWR_OUT = OUTPUT / "MGWR"
MGWR_OUT.mkdir(
    parents=True,
    exist_ok=True
)

FEATURES = [
    "TEMP_C",
    "SW_RAD_WM2",
    "RH",
    "WIND_MS",
    "PRECIP_MM",
    "NDVI",
    "SOIL_MM",
    "ELEV_M"
]

SHORT = {
    "TEMP_C":
        ["TEMP", "TEMPC"],

    "SW_RAD_WM2":
        ["SWRAD", "SWRADWM2"],

    "RH":
        ["RH"],

    "WIND_MS":
        ["WIND", "WINDMS"],

    "PRECIP_MM":
        ["PRECIP", "PRECIPMM"],

    "NDVI":
        ["NDVI"],

    "SOIL_MM":
        ["SOIL", "SOILMM"],

    "ELEV_M":
        ["ELEV", "ELEVM"],
}


# ==========================================================================================
# HELPER — coefficient column detection
# ==========================================================================================

def find_beta_column(df, feature):

    aliases = [
        feature,
        *SHORT.get(
            feature,
            []
        )
    ]

    exact = []

    for a in aliases:

        exact += [
            f"BETA_{a}",
            f"B_{a}",
            f"COEF_{a}",
            f"LOCAL_BETA_{a}",
            f"MGWR_BETA_{a}",
        ]

    # Exact normalized search
    colnorm = {
        norm_name(c): c
        for c in df.columns
    }

    for x in exact:

        nx = norm_name(x)

        if nx in colnorm:
            return colnorm[nx]

    # Fuzzy fallback
    for c in df.columns:

        nc = norm_name(c)

        if not (
            "BETA" in nc
            or nc.startswith("B")
            or "COEF" in nc
        ):
            continue

        for a in aliases:

            if norm_name(a) in nc:
                return c

    raise KeyError(
        f"Could not find MGWR beta for {feature}\n"
        f"Columns:\n{list(df.columns)}"
    )


def find_intercept_column(df):

    candidates = [
        "BETA_INTERCEPT",
        "B_INTERCEPT",
        "BETA_INT",
        "B_INT",
        "COEF_INTERCEPT",
        "INTERCEPT_BETA",
        "LOCAL_INTERCEPT"
    ]

    colnorm = {
        norm_name(c): c
        for c in df.columns
    }

    for c in candidates:

        if norm_name(c) in colnorm:
            return colnorm[
                norm_name(c)
            ]

    for c in df.columns:

        nc = norm_name(c)

        if (
            "INTERCEPT" in nc
            and
            (
                "BETA" in nc
                or nc.startswith("B")
                or "COEF" in nc
            )
        ):
            return c

    raise KeyError(
        "MGWR intercept coefficient not found.\n"
        f"Columns:\n{list(df.columns)}"
    )


# ==========================================================================================
# PART A — REAL MGWR, 2015–2024
# ==========================================================================================

print("\n" + "=" * 110)
print("PART A — REAL MGWR 2015–2024")
print("=" * 110)

REAL_PANEL = get_real_panel()

REAL_PANEL = REAL_PANEL.loc[
    REAL_PANEL[
        "YEAR"
    ].isin(
        YEARS
    )
].copy()

REAL_PANEL[
    "HEX_ID"
] = REAL_PANEL[
    "HEX_ID"
].astype(str)

# coordinate aliases
rx = choose_column(
    REAL_PANEL,
    [
        "X_KM",
        "CX_KM"
    ]
)

ry = choose_column(
    REAL_PANEL,
    [
        "Y_KM",
        "CY_KM"
    ]
)

# ----------------------------------------------------------------------
# Locate FULL MGWR local coefficient table
# ----------------------------------------------------------------------

real_mgwr = None
real_mgwr_path = None

preferred_names = [
    "MGWR_LOCAL_RESULTS.csv",
    "MGWR_LOCAL_RESULTS.parquet",
    "MGWR_FULL_LOCAL_RESULTS.csv",
    "MGWR_FULL_LOCAL_RESULTS.parquet",
    "MGWR_LOCAL_ALL_HEX.parquet"
]

for p in TABLE_FILES:

    ps = str(p).upper()

    if "MGWR" not in ps:
        continue

    if "SIMULATION" in ps:
        continue

    if p.name.upper() not in {
        x.upper()
        for x in preferred_names
    }:
        continue

    try:
        q = read_table(p)

    except Exception:
        continue

    if "HEX_ID" not in q.columns:
        continue

    try:

        _ = find_intercept_column(
            q
        )

        for f in FEATURES:
            _ = find_beta_column(
                q,
                f
            )

        real_mgwr = q
        real_mgwr_path = p
        break

    except Exception:
        continue


# broader fallback
if real_mgwr is None:

    candidates = [
        p for p in TABLE_FILES
        if "MGWR" in str(p).upper()
        and "SIMULATION" not in str(p).upper()
        and (
            "LOCAL" in p.name.upper()
            or "RESULT" in p.name.upper()
        )
    ]

    for p in candidates:

        try:
            q = read_table(p)

        except Exception:
            continue

        if "HEX_ID" not in q.columns:
            continue

        try:

            _ = find_intercept_column(
                q
            )

            for f in FEATURES:
                _ = find_beta_column(
                    q,
                    f
                )

            real_mgwr = q
            real_mgwr_path = p
            break

        except Exception:
            continue


if real_mgwr is None:

    raise RuntimeError(
        "\nREAL MGWR local coefficient table not found.\n"
        "Expected the FULL-CONUS MGWR output containing HEX_ID + "
        "intercept + 8 local beta columns."
    )

print(
    "REAL MGWR coefficients:",
    real_mgwr_path
)

real_mgwr[
    "HEX_ID"
] = real_mgwr[
    "HEX_ID"
].astype(str)

b0_col = find_intercept_column(
    real_mgwr
)

beta_cols = {
    f: find_beta_column(
        real_mgwr,
        f
    )
    for f in FEATURES
}

print("\nCoefficient mapping:")
print("Intercept ->", b0_col)

for k, v in beta_cols.items():
    print(k, "->", v)


# ----------------------------------------------------------------------
# Reproduce the SAME climatological standardization used by full MGWR
#
# This is NOT an annual MGWR fit.
# One fixed MGWR coefficient surface is applied to yearly covariates.
# ----------------------------------------------------------------------

mg_ids = set(
    real_mgwr[
        "HEX_ID"
    ]
)

RP = REAL_PANEL.loc[
    REAL_PANEL[
        "HEX_ID"
    ].isin(
        mg_ids
    )
].copy()

clim = (
    RP.groupby(
        "HEX_ID",
        as_index=False
    )[FEATURES]
    .mean()
)

means = clim[
    FEATURES
].mean()

scales = clim[
    FEATURES
].std(
        ddof=0
    )

scales = scales.replace(
    0,
    np.nan
)

coef_keep = [
    "HEX_ID",
    b0_col,
    *beta_cols.values()
]

coef = (
    real_mgwr[
        coef_keep
    ]
    .drop_duplicates(
        "HEX_ID"
    )
)

RP = RP.merge(
    coef,
    on="HEX_ID",
    how="inner",
    validate="m:1"
)

RP[
    "MGWR_MODEL_LST_C"
] = pd.to_numeric(
    RP[
        b0_col
    ],
    errors="coerce"
)

for f in FEATURES:

    z = (
        pd.to_numeric(
            RP[f],
            errors="coerce"
        )
        -
        means[f]
    ) / scales[f]

    RP[
        "MGWR_MODEL_LST_C"
    ] += (
        pd.to_numeric(
            RP[
                beta_cols[f]
            ],
            errors="coerce"
        )
        *
        z
    )

print(
    "\nAnnual MGWR predictions created:",
    f"{RP['MGWR_MODEL_LST_C'].notna().sum():,}"
)

MGWR_REAL, MGWR_REAL_CUTS, MGWR_REAL_METRICS = build_model_gvstmr(
    df=RP,
    id_col="HEX_ID",
    time_col="YEAR",
    model_col="MGWR_MODEL_LST_C",
    obs_col="LST_MEAN_C",
    x_col=rx,
    y_col=ry,
    case_name="REAL",
    model_name="MGWR",
    output_dir=MGWR_OUT / "REAL"
)


# ==========================================================================================
# PART B — SIMULATION MGWR, T1–T10
# ==========================================================================================

print("\n" + "=" * 110)
print("PART B — SIMULATION MGWR T1–T10")
print("=" * 110)

SIM = get_sim_panel()

SIM[
    "HEX_ID"
] = SIM[
    "HEX_ID"
].astype(str)

# Locate original simulation MGWR local output
sim_mgwr, sim_mgwr_path = locate_exact_table(
    [
        "MGWR_LOCAL_ALL_HEX.parquet"
    ],
    required_cols=[
        "HEX_ID",
        "BETA_INTERCEPT",
        "BETA_X"
    ],
    prefer_terms=[
        "SIMULATION",
        "03_MGWR"
    ]
)

if sim_mgwr is None:

    raise RuntimeError(
        "Simulation MGWR_LOCAL_ALL_HEX.parquet not found."
    )

sim_mgwr[
    "HEX_ID"
] = sim_mgwr[
    "HEX_ID"
].astype(str)

sim_ids = set(
    sim_mgwr[
        "HEX_ID"
    ]
)

SP = SIM.loc[
    SIM[
        "HEX_ID"
    ].isin(
        sim_ids
    )
].copy()

# Same scaler used by climatological simulation MGWR:
# first climatological X for each hex, then global StandardScaler.
sim_clim_x = (
    SP.groupby(
        "HEX_ID"
    )[
        "X"
    ]
    .mean()
)

sim_x_mean = float(
    sim_clim_x.mean()
)

sim_x_scale = float(
    sim_clim_x.std(
        ddof=0
    )
)

if not np.isfinite(
    sim_x_scale
) or sim_x_scale <= 0:

    raise RuntimeError(
        "Invalid simulation climatological X scale."
    )

SP = SP.merge(
    sim_mgwr[
        [
            "HEX_ID",
            "BETA_INTERCEPT",
            "BETA_X"
        ]
    ],
    on="HEX_ID",
    how="inner",
    validate="m:1"
)

SP[
    "X_MGWR_Z"
] = (
    SP[
        "X"
    ].to_numpy(float)
    -
    sim_x_mean
) / sim_x_scale

SP[
    "MGWR_MODEL_Y"
] = (
    SP[
        "BETA_INTERCEPT"
    ].to_numpy(float)
    +
    SP[
        "BETA_X"
    ].to_numpy(float)
    *
    SP[
        "X_MGWR_Z"
    ].to_numpy(float)
)

MGWR_SIM, MGWR_SIM_CUTS, MGWR_SIM_METRICS = build_model_gvstmr(
    df=SP,
    id_col="HEX_ID",
    time_col="TIME",
    model_col="MGWR_MODEL_Y",
    obs_col="Y",
    x_col="X_KM",
    y_col="Y_KM",
    case_name="SIMULATION",
    model_name="MGWR",
    output_dir=MGWR_OUT / "SIMULATION",
    keep_cols=[
        "Y_SIGNAL",
        "TRUE_BETA_X"
    ]
)

# Save model prediction panel for later cartography
SP[
    [
        "HEX_ID",
        "TIME",
        "X_KM",
        "Y_KM",
        "X",
        "Y",
        "Y_SIGNAL",
        "TRUE_BETA_X",
        "MGWR_MODEL_Y"
    ]
].to_parquet(
    MGWR_OUT
    /
    "SIMULATION"
    /
    "MGWR_SIM_T1_T10_PREDICTIONS.parquet",
    index=False
)

print("\n" + "=" * 110)
print("✅ MGWR REAL + SIMULATION GVSTMR COMPLETE")
print("=" * 110)
print("Output:", MGWR_OUT)
print("NO MGWR ANNUAL REFIT.")
print("NO GOOGLE DRIVE WRITE.")

In [ ]:
# ==========================================================================================
# CELL 2 — GTWR → GVSTMR
#
# REAL       : 2015–2024
# SIMULATION : T1–T10
#
# NO GTWR REFIT
# NO GOOGLE DRIVE WRITE
# ==========================================================================================

print("=" * 110)
print("GTWR → MODEL-vs-OBSERVATION GVSTMR")
print("=" * 110)

GTWR_OUT = OUTPUT / "GTWR"
GTWR_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ==========================================================================================
# HELPER — find prediction column
# ==========================================================================================

def detect_prediction_column(
    df,
    model_name,
    preferred
):

    for c in preferred:

        if c in df.columns:
            return c

    cols = list(
        df.columns
    )

    # Prefer explicit MODEL + PRED
    for c in cols:

        u = c.upper()

        if (
            model_name.upper() in u
            and
            (
                "PRED" in u
                or "FITTED" in u
            )
        ):
            return c

    for c in cols:

        u = c.upper()

        if (
            "PRED" in u
            or "FITTED" in u
        ):
            return c

    raise KeyError(
        f"{model_name} prediction column not found.\n"
        f"Columns:\n{list(df.columns)}"
    )


# ==========================================================================================
# PART A — REAL GTWR
# ==========================================================================================

print("\n" + "=" * 110)
print("PART A — REAL GTWR 2015–2024")
print("=" * 110)

REAL_PANEL = get_real_panel()

REAL_PANEL = REAL_PANEL.loc[
    REAL_PANEL[
        "YEAR"
    ].isin(
        YEARS
    )
].copy()

REAL_PANEL[
    "HEX_ID"
] = REAL_PANEL[
    "HEX_ID"
].astype(str)

rx = choose_column(
    REAL_PANEL,
    [
        "X_KM",
        "CX_KM"
    ]
)

ry = choose_column(
    REAL_PANEL,
    [
        "Y_KM",
        "CY_KM"
    ]
)

# Search real GTWR table
real_gtwr = None
real_gtwr_path = None

candidate_gtwr = [
    p for p in TABLE_FILES
    if "GTWR" in str(p).upper()
    and "SIMULATION" not in str(p).upper()
    and (
        "FULL" in str(p).upper()
        or "FINAL" in str(p).upper()
        or "RESULT" in str(p).upper()
    )
]

# Prefer large/full panel files
candidate_gtwr = sorted(
    candidate_gtwr,
    key=lambda p: (
        "GTWR_FULL" in str(p).upper(),
        "RESULT" in p.name.upper(),
        p.stat().st_size
        if p.exists()
        else 0
    ),
    reverse=True
)

for p in candidate_gtwr:

    try:
        q = read_table(p)

    except Exception:
        continue

    if not {
        "HEX_ID",
        "YEAR"
    }.issubset(
        q.columns
    ):
        continue

    try:

        pc = detect_prediction_column(
            q,
            "GTWR",
            [
                "GTWR_PRED_LST_C",
                "GTWR_PRED_C",
                "GTWR_PRED",
                "PRED_LST_C",
                "PRED",
                "FITTED_LST_C",
                "FITTED"
            ]
        )

    except Exception:
        continue

    real_gtwr = q
    real_gtwr_path = p
    real_pred_col = pc
    break


if real_gtwr is None:

    # runtime object fallback
    for objname in [
        "GTWR_FULL",
        "GTWR_ALL",
        "GT"
    ]:

        if objname in globals():

            q = globals()[
                objname
            ]

            if not isinstance(
                q,
                pd.DataFrame
            ):
                continue

            if not {
                "HEX_ID",
                "YEAR"
            }.issubset(
                q.columns
            ):
                continue

            try:

                pc = detect_prediction_column(
                    q,
                    "GTWR",
                    [
                        "GTWR_PRED_LST_C",
                        "GTWR_PRED_C",
                        "GTWR_PRED",
                        "PRED",
                        "FITTED"
                    ]
                )

                real_gtwr = q.copy()
                real_gtwr_path = (
                    f"runtime:{objname}"
                )
                real_pred_col = pc
                break

            except Exception:
                pass


if real_gtwr is None:

    raise RuntimeError(
        "\nREAL full GTWR table not found.\n"
        "Need HEX_ID + YEAR + GTWR prediction."
    )

print(
    "REAL GTWR:",
    real_gtwr_path
)

print(
    "Prediction column:",
    real_pred_col
)

real_gtwr[
    "HEX_ID"
] = real_gtwr[
    "HEX_ID"
].astype(str)

real_gtwr = real_gtwr.loc[
    real_gtwr[
        "YEAR"
    ].isin(
        YEARS
    )
].copy()

# Add observed LST / coords if missing
need_from_panel = [
    "LST_MEAN_C"
]

if rx not in real_gtwr.columns:
    need_from_panel.append(
        rx
    )

if ry not in real_gtwr.columns:
    need_from_panel.append(
        ry
    )

if any(
    c not in real_gtwr.columns
    for c in need_from_panel
):

    panel_add = (
        REAL_PANEL[
            [
                "HEX_ID",
                "YEAR",
                "LST_MEAN_C",
                rx,
                ry
            ]
        ]
        .drop_duplicates(
            [
                "HEX_ID",
                "YEAR"
            ]
        )
    )

    # Only columns not already present
    addcols = [
        c for c in [
            "LST_MEAN_C",
            rx,
            ry
        ]
        if c not in real_gtwr.columns
    ]

    real_gtwr = real_gtwr.merge(
        panel_add[
            [
                "HEX_ID",
                "YEAR",
                *addcols
            ]
        ],
        on=[
            "HEX_ID",
            "YEAR"
        ],
        how="left",
        validate="1:1"
    )

gx = (
    rx
    if rx in real_gtwr.columns
    else choose_column(
        real_gtwr,
        [
            "X_KM",
            "CX_KM"
        ]
    )
)

gy = (
    ry
    if ry in real_gtwr.columns
    else choose_column(
        real_gtwr,
        [
            "Y_KM",
            "CY_KM"
        ]
    )
)

GTWR_REAL, GTWR_REAL_CUTS, GTWR_REAL_METRICS = build_model_gvstmr(
    df=real_gtwr,
    id_col="HEX_ID",
    time_col="YEAR",
    model_col=real_pred_col,
    obs_col="LST_MEAN_C",
    x_col=gx,
    y_col=gy,
    case_name="REAL",
    model_name="GTWR",
    output_dir=GTWR_OUT / "REAL"
)


# ==========================================================================================
# PART B — SIMULATION GTWR
# ==========================================================================================

print("\n" + "=" * 110)
print("PART B — SIMULATION GTWR T1–T10")
print("=" * 110)

sim_gtwr, sim_gtwr_path = locate_exact_table(
    [
        "GTWR_ALL_HEX_T1_T10.parquet"
    ],
    required_cols=[
        "HEX_ID",
        "TIME",
        "Y",
        "GTWR_PRED_Y"
    ],
    prefer_terms=[
        "SIMULATION",
        "04_GTWR"
    ]
)

# Fallback — combine cartography T1...T10 files
if sim_gtwr is None:

    pieces = []

    for t in SIM_TIMES:

        q, qp = locate_exact_table(
            [
                f"GTWR_SIM_T{t}.parquet"
            ],
            required_cols=[
                "HEX_ID",
                "GTWR_PRED_Y"
            ],
            prefer_terms=[
                "SIMULATION"
            ]
        )

        if q is None:
            pieces = []
            break

        if "TIME" not in q.columns:
            q[
                "TIME"
            ] = t

        pieces.append(
            q
        )

    if pieces:

        sim_gtwr = pd.concat(
            pieces,
            ignore_index=True
        )

        sim_gtwr_path = (
            "combined GTWR_SIM_T1...T10"
        )


if sim_gtwr is None:

    raise RuntimeError(
        "Simulation GTWR_ALL_HEX_T1_T10.parquet not found."
    )

print(
    "Simulation GTWR:",
    sim_gtwr_path
)

sim_gtwr[
    "HEX_ID"
] = sim_gtwr[
    "HEX_ID"
].astype(str)

# Ensure coords/Y exist by merging SIM_PANEL if necessary
SIM = get_sim_panel()

SIM[
    "HEX_ID"
] = SIM[
    "HEX_ID"
].astype(str)

required_sim_fields = [
    "Y",
    "X_KM",
    "Y_KM"
]

missing = [
    c for c in required_sim_fields
    if c not in sim_gtwr.columns
]

if missing:

    sim_gtwr = sim_gtwr.merge(
        SIM[
            [
                "HEX_ID",
                "TIME",
                *missing
            ]
        ],
        on=[
            "HEX_ID",
            "TIME"
        ],
        how="left",
        validate="1:1"
    )

GTWR_SIM, GTWR_SIM_CUTS, GTWR_SIM_METRICS = build_model_gvstmr(
    df=sim_gtwr,
    id_col="HEX_ID",
    time_col="TIME",
    model_col="GTWR_PRED_Y",
    obs_col="Y",
    x_col="X_KM",
    y_col="Y_KM",
    case_name="SIMULATION",
    model_name="GTWR",
    output_dir=GTWR_OUT / "SIMULATION",
    keep_cols=[
        "BETA_INTERCEPT",
        "BETA_X",
        "LOCAL_ST_BW"
    ]
)

print("\n" + "=" * 110)
print("✅ GTWR REAL + SIMULATION GVSTMR COMPLETE")
print("=" * 110)
print("Output:", GTWR_OUT)
print("NO GTWR REFIT.")
print("NO GOOGLE DRIVE WRITE.")

In [ ]:
# ==========================================================================================
# CELL 3 — GGPR → GVSTMR
#
# REAL       : existing annual GGPR predictions 2015–2024
# SIMULATION : exact final GGPR reconstructed from SAVED calibration + SAVED hyperparameters
#              then applied to T1–T10
#
# NO HYPERPARAMETER TUNING
# NO GOOGLE DRIVE WRITE
# ==========================================================================================

print("=" * 110)
print("GGPR → MODEL-vs-OBSERVATION GVSTMR")
print("=" * 110)

from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Kernel

GGPR_OUT = OUTPUT / "GGPR"
GGPR_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ==========================================================================================
# PART A — REAL ANNUAL GGPR 2015–2024
# ==========================================================================================

print("\n" + "=" * 110)
print("PART A — REAL GGPR 2015–2024")
print("=" * 110)

REAL_GG, REAL_GG_PATH = locate_exact_table(
    [
        "ANNUAL_GGPR_GEOSHAP_UNCERTAINTY_ALL_HEX_2015_2024.parquet",
        "ANNUAL_GGPR_GEOSHAP_UNCERTAINTY_COUPLING_ALL_HEX.parquet"
    ],
    required_cols=[
        "HEX_ID",
        "YEAR",
        "LST_MEAN_C",
        "GGPR_PRED_LST_C"
    ],
    prefer_terms=[
        "ANNUAL_GGPR",
        "2015_2024"
    ]
)

# fallback: combine ten annual files
if REAL_GG is None:

    parts = []

    for year in YEARS:

        q, qp = locate_exact_table(
            [
                f"ANNUAL_GGPR_GEOSHAP_{year}.parquet",
                f"ANNUAL_GGPR_GEOSHAP_UNCERTAINTY_{year}.parquet"
            ],
            required_cols=[
                "HEX_ID",
                "YEAR",
                "LST_MEAN_C",
                "GGPR_PRED_LST_C"
            ],
            prefer_terms=[
                "ANNUAL_GGPR"
            ]
        )

        if q is None:

            parts = []
            break

        parts.append(
            q
        )

    if parts:

        REAL_GG = pd.concat(
            parts,
            ignore_index=True
        )

        REAL_GG_PATH = (
            "combined annual GGPR 2015–2024"
        )


if REAL_GG is None:

    raise RuntimeError(
        "\nAnnual real GGPR output not found.\n"
        "Upload:\n"
        "GVSTMR_ANNUAL_GGPR_GEOSHAP_UNCERTAINTY_2015_2024.zip"
    )

print(
    "REAL GGPR:",
    REAL_GG_PATH
)

REAL_GG[
    "HEX_ID"
] = REAL_GG[
    "HEX_ID"
].astype(str)

REAL_GG = REAL_GG.loc[
    REAL_GG[
        "YEAR"
    ].isin(
        YEARS
    )
].copy()

# Add coordinates if annual table does not contain them
REAL_PANEL = get_real_panel()

REAL_PANEL[
    "HEX_ID"
] = REAL_PANEL[
    "HEX_ID"
].astype(str)

rx = choose_column(
    REAL_PANEL,
    [
        "X_KM",
        "CX_KM"
    ]
)

ry = choose_column(
    REAL_PANEL,
    [
        "Y_KM",
        "CY_KM"
    ]
)

missing_coords = [
    c for c in [
        rx,
        ry
    ]
    if c not in REAL_GG.columns
]

if missing_coords:

    coords_add = (
        REAL_PANEL[
            [
                "HEX_ID",
                "YEAR",
                rx,
                ry
            ]
        ]
        .drop_duplicates(
            [
                "HEX_ID",
                "YEAR"
            ]
        )
    )

    REAL_GG = REAL_GG.merge(
        coords_add[
            [
                "HEX_ID",
                "YEAR",
                *missing_coords
            ]
        ],
        on=[
            "HEX_ID",
            "YEAR"
        ],
        how="left",
        validate="1:1"
    )

gx = (
    rx
    if rx in REAL_GG.columns
    else choose_column(
        REAL_GG,
        [
            "X_KM",
            "CX_KM"
        ]
    )
)

gy = (
    ry
    if ry in REAL_GG.columns
    else choose_column(
        REAL_GG,
        [
            "Y_KM",
            "CY_KM"
        ]
    )
)

# Preserve uncertainty + GeoShapley columns for later cartography
gg_keep = [
    c for c in REAL_GG.columns
    if (
        c.startswith(
            "GEOSHAP_"
        )
        or c in [
            "GGPR_POSTERIOR_SD_C",
            "GGPR_CI95_WIDTH_C",
            "GGPR_RESID_C"
        ]
    )
]

GGPR_REAL, GGPR_REAL_CUTS, GGPR_REAL_METRICS = build_model_gvstmr(
    df=REAL_GG,
    id_col="HEX_ID",
    time_col="YEAR",
    model_col="GGPR_PRED_LST_C",
    obs_col="LST_MEAN_C",
    x_col=gx,
    y_col=gy,
    case_name="REAL",
    model_name="GGPR",
    output_dir=GGPR_OUT / "REAL",
    keep_cols=gg_keep
)


# ==========================================================================================
# PART B — SIMULATION GGPR T1–T10
# ==========================================================================================

print("\n" + "=" * 110)
print("PART B — SIMULATION GGPR T1–T10")
print("=" * 110)

SIM = get_sim_panel()

SIM[
    "HEX_ID"
] = SIM[
    "HEX_ID"
].astype(str)


# ==========================================================================================
# First check whether time-specific GGPR predictions already exist
# ==========================================================================================

SIM_GG = None
SIM_GG_SOURCE = None

sim_candidates = [
    p for p in TABLE_FILES
    if "GGPR" in str(p).upper()
    and "SIMULATION" in str(p).upper()
]

for p in sim_candidates:

    try:
        q = read_table(
            p
        )

    except Exception:
        continue

    if not {
        "HEX_ID",
        "TIME",
        "GGPR_PRED_Y"
    }.issubset(
        q.columns
    ):
        continue

    if q[
        "TIME"
    ].nunique() < 10:
        continue

    SIM_GG = q.copy()
    SIM_GG_SOURCE = p
    break


# ==========================================================================================
# If not saved, reconstruct EXACT final simulation GGPR from saved calibration.
#
# IMPORTANT:
# - NO hyperparameter search
# - NO sample-size sensitivity
# - NO cross-validation
# - NO GeoShapley rerun
#
# It only rebuilds the final GP object deterministically and evaluates T1–T10.
# ==========================================================================================

if SIM_GG is None:

    print(
        "\nNo saved T1–T10 GGPR predictions found."
    )

    print(
        "Reconstructing exact final GGPR from saved calibration + selected hyperparameters..."
    )

    CAL, CAL_PATH = locate_exact_table(
        [
            "GGPR_FINAL_CALIBRATION.csv"
        ],
        required_cols=[
            "HEX_ID",
            "X_KM",
            "Y_KM",
            "X",
            "Y"
        ],
        prefer_terms=[
            "SIMULATION",
            "05_GGPR"
        ]
    )

    TUNE, TUNE_PATH = locate_exact_table(
        [
            "GGPR_HYPERPARAMETER_TUNING.csv"
        ],
        required_cols=[
            "SPATIAL_WEIGHT",
            "LENGTH_SCALE"
        ],
        prefer_terms=[
            "SIMULATION",
            "05_GGPR"
        ]
    )

    if CAL is None:

        raise RuntimeError(
            "Simulation GGPR_FINAL_CALIBRATION.csv not found."
        )

    if TUNE is None:

        raise RuntimeError(
            "Simulation GGPR_HYPERPARAMETER_TUNING.csv not found."
        )

    print(
        "Calibration:",
        CAL_PATH
    )

    print(
        "Tuning table:",
        TUNE_PATH
    )

    # The original tuning table was saved sorted by RMSE ascending.
    if "RMSE" in TUNE.columns:

        best = (
            TUNE.sort_values(
                [
                    "RMSE",
                    "R2"
                ],
                ascending=[
                    True,
                    False
                ]
            )
            .iloc[0]
        )

    else:

        best = TUNE.iloc[
            0
        ]

    BEST_SW = float(
        best[
            "SPATIAL_WEIGHT"
        ]
    )

    BEST_LS = float(
        best[
            "LENGTH_SCALE"
        ]
    )

    print(
        "Final spatial weight:",
        BEST_SW
    )

    print(
        "Final length scale :",
        BEST_LS
    )

    # Original V2 simulation setting
    GGPR_NOISE = 0.03
    RANDOM_STATE = 123


    # ======================================================================================
    # EXACT original Simulation GGPR kernel
    # ======================================================================================

    class SimulationGGPRKernel(Kernel):

        def __init__(
            self,
            spatial_weight=0.10,
            length_scale=1.0
        ):

            self.spatial_weight = spatial_weight
            self.length_scale = length_scale


        def __call__(
            self,
            X,
            Y=None,
            eval_gradient=False
        ):

            X = np.asarray(
                X,
                float
            )

            if Y is None:

                Y = X
                same = True

            else:

                Y = np.asarray(
                    Y,
                    float
                )

                same = False

            # Feature similarity component
            d_feature = (
                X[:, 0][:, None]
                -
                Y[:, 0][None, :]
            )

            ssk = np.exp(
                -0.5
                *
                d_feature**2
            )

            # Geographic Matern 3/2 component
            dx = (
                X[:, 1][:, None]
                -
                Y[:, 1][None, :]
            )

            dy = (
                X[:, 2][:, None]
                -
                Y[:, 2][None, :]
            )

            distance = np.sqrt(
                dx**2
                +
                dy**2
            )

            r = (
                np.sqrt(
                    3.0
                )
                *
                distance
                /
                float(
                    self.length_scale
                )
            )

            matern = (
                1.0
                +
                r
            ) * np.exp(
                -r
            )

            K = (
                ssk
                +
                float(
                    self.spatial_weight
                )
                *
                matern
            )

            if eval_gradient:

                if not same:

                    raise ValueError(
                        "Gradient only supported for Y=None."
                    )

                return (
                    K,
                    np.empty(
                        (
                            len(X),
                            len(X),
                            0
                        )
                    )
                )

            return K


        def diag(
            self,
            X
        ):

            return np.full(
                len(X),
                1.0
                +
                float(
                    self.spatial_weight
                )
            )


        def is_stationary(
            self
        ):

            return True


    # ======================================================================================
    # Reconstruct scalers from SAME saved final calibration sample
    # ======================================================================================

    x_scaler = StandardScaler()

    XZ_CAL = x_scaler.fit_transform(
        CAL[
            ["X"]
        ]
    )

    coord_scaler = StandardScaler()

    CZ_CAL = coord_scaler.fit_transform(
        CAL[
            [
                "X_KM",
                "Y_KM"
            ]
        ]
    )

    y_scaler = StandardScaler()

    YZ_CAL = y_scaler.fit_transform(
        CAL[
            ["Y"]
        ]
    ).ravel()

    A_CAL = np.column_stack(
        [
            XZ_CAL,
            CZ_CAL
        ]
    )

    kernel = SimulationGGPRKernel(
        spatial_weight=BEST_SW,
        length_scale=BEST_LS
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=GGPR_NOISE,
        optimizer=None,
        normalize_y=False,
        random_state=RANDOM_STATE
    )

    print(
        "\nRebuilding final 900-point GP object..."
    )

    gp.fit(
        A_CAL,
        YZ_CAL
    )

    print(
        "Final GGPR reconstructed."
    )


    # ======================================================================================
    # Predict all Hex × T1–T10
    # ======================================================================================

    SIM_GG = SIM[
        [
            "HEX_ID",
            "TIME",
            "X_KM",
            "Y_KM",
            "X",
            "Y",
            "Y_SIGNAL",
            "TRUE_BETA_X"
        ]
    ].copy()

    PRED = np.full(
        len(SIM_GG),
        np.nan
    )

    SD = np.full(
        len(SIM_GG),
        np.nan
    )

    BATCH = 1200

    for start in range(
        0,
        len(SIM_GG),
        BATCH
    ):

        end = min(
            start
            +
            BATCH,
            len(SIM_GG)
        )

        q = SIM_GG.iloc[
            start:end
        ]

        qx = x_scaler.transform(
            q[
                ["X"]
            ]
        )

        qc = coord_scaler.transform(
            q[
                [
                    "X_KM",
                    "Y_KM"
                ]
            ]
        )

        A = np.column_stack(
            [
                qx,
                qc
            ]
        )

        mean_z, std_z = gp.predict(
            A,
            return_std=True
        )

        mean_y = (
            y_scaler
            .inverse_transform(
                mean_z.reshape(
                    -1,
                    1
                )
            )
            .ravel()
        )

        std_y = (
            std_z
            *
            float(
                y_scaler.scale_[
                    0
                ]
            )
        )

        PRED[
            start:end
        ] = mean_y

        SD[
            start:end
        ] = std_y

        if (
            end % 12000 < BATCH
            or end == len(
                SIM_GG
            )
        ):

            print(
                f"GGPR prediction: {end:,}/{len(SIM_GG):,}"
            )

    SIM_GG[
        "GGPR_PRED_Y"
    ] = PRED

    SIM_GG[
        "GGPR_POSTERIOR_SD"
    ] = SD

    SIM_GG[
        "GGPR_CI95_LO"
    ] = (
        PRED
        -
        1.96
        *
        SD
    )

    SIM_GG[
        "GGPR_CI95_HI"
    ] = (
        PRED
        +
        1.96
        *
        SD
    )

    SIM_GG[
        "GGPR_CI95_WIDTH"
    ] = (
        3.92
        *
        SD
    )

    SIM_GG[
        "GGPR_RESID"
    ] = (
        SIM_GG[
            "Y"
        ]
        -
        PRED
    )

    SIM_GG_SOURCE = (
        "reconstructed exact final GGPR "
        "from saved calibration/hyperparameters"
    )

    # Save once so future cartography does NOT need to rebuild GP
    SIM_GG.to_parquet(
        GGPR_OUT
        /
        "SIMULATION_GGPR_T1_T10_PREDICTIONS.parquet",
        index=False
    )

    print(
        "\n✅ T1–T10 GGPR predictions saved locally."
    )


else:

    print(
        "Using existing Simulation GGPR:",
        SIM_GG_SOURCE
    )

    SIM_GG[
        "HEX_ID"
    ] = SIM_GG[
        "HEX_ID"
    ].astype(str)

    # Add missing observed/coordinates from SIM panel
    missing = [
        c for c in [
            "Y",
            "X_KM",
            "Y_KM"
        ]
        if c not in SIM_GG.columns
    ]

    if missing:

        SIM_GG = SIM_GG.merge(
            SIM[
                [
                    "HEX_ID",
                    "TIME",
                    *missing
                ]
            ],
            on=[
                "HEX_ID",
                "TIME"
            ],
            how="left",
            validate="1:1"
        )


# ==========================================================================================
# SIMULATION GGPR → GVSTMR
# ==========================================================================================

GGPR_SIM, GGPR_SIM_CUTS, GGPR_SIM_METRICS = build_model_gvstmr(
    df=SIM_GG,
    id_col="HEX_ID",
    time_col="TIME",
    model_col="GGPR_PRED_Y",
    obs_col="Y",
    x_col="X_KM",
    y_col="Y_KM",
    case_name="SIMULATION",
    model_name="GGPR",
    output_dir=GGPR_OUT / "SIMULATION",
    keep_cols=[
        "GGPR_POSTERIOR_SD",
        "GGPR_CI95_LO",
        "GGPR_CI95_HI",
        "GGPR_CI95_WIDTH",
        "GGPR_RESID",
        "Y_SIGNAL",
        "TRUE_BETA_X"
    ]
)

print("\n" + "=" * 110)
print("✅ GGPR REAL + SIMULATION GVSTMR COMPLETE")
print("=" * 110)
print("Output:", GGPR_OUT)
print("No hyperparameter tuning was rerun.")
print("NO GOOGLE DRIVE WRITE.")

gc.collect()

## Fresh-runtime bootstrap complete

The next cells generate only supplementary figures/tables.  
Do not restart the runtime between the bootstrap and supplementary cells.


In [ ]:
# ================================================================================================
# SUPPLEMENTARY SETUP — RUN AFTER THE FINAL MAIN-FIGURE NOTEBOOK
# ================================================================================================

from pathlib import Path
import warnings
import shutil
import zipfile
import math
import re
import json

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.patches import Rectangle
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, Normalize
from scipy.spatial import cKDTree
from scipy.stats import spearmanr
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------------------------------------
# REQUIRED FINAL OBJECTS FROM THE MAIN NOTEBOOK
# ------------------------------------------------------------------------------------------------

REQUIRED_CORE = [
    "SIM_DIRECT",
    "REAL_DIRECT",
    "SIM_HEX",
    "HEX",
    "SIM_PANEL",
    "PANEL",
    "GV_PALETTE",
    "SIM_TIMES",
    "YEARS",
]

missing_core = [
    name
    for name in REQUIRED_CORE
    if name not in globals()
]

if missing_core:
    raise RuntimeError(
        "Fresh-runtime bootstrap did not finish successfully.\n"
        "Missing required objects:\n - " + "\n - ".join(missing_core)
    )

# ------------------------------------------------------------------------------------------------
# OUTPUT ROOT — NEVER TOUCH THE MAIN-FIGURE DIRECTORY
# ------------------------------------------------------------------------------------------------

SUP_ROOT = Path("/content/GVSTMR_SUPPLEMENTARY")
SUP_FIG = SUP_ROOT / "FIGURES"
SUP_DATA = SUP_ROOT / "DATA"
SUP_META = SUP_ROOT / "METADATA"

SUP_ZIP = Path("/content/GVSTMR_SUPPLEMENTARY_OUTPUTS.zip")

if SUP_ROOT.exists():
    shutil.rmtree(SUP_ROOT)

if SUP_ZIP.exists():
    SUP_ZIP.unlink()

for p in [SUP_FIG, SUP_DATA, SUP_META]:
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------------------------
# SAME PUBLICATION STYLE + SAME GVSTMR PALETTE
# ------------------------------------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 9.0,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.4,
    "ytick.labelsize": 7.4,
    "legend.fontsize": 7.2,
    "figure.titlesize": 11.0,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

GV = dict(GV_PALETTE)

MODEL_COLORS = {
    "MGWR": GV[7],
    "GTWR": GV[6],
    "GGPR": GV[9],
}

COEF_CMAP = LinearSegmentedColormap.from_list(
    "GVSTMR_coef",
    [GV[9], "#f7f7f7", GV[7]]
)

UNC_CMAP = LinearSegmentedColormap.from_list(
    "GVSTMR_unc",
    [GV[3], GV[2], GV[5], GV[8], GV[7]]
)

# ------------------------------------------------------------------------------------------------
# GENERAL HELPERS
# ------------------------------------------------------------------------------------------------

def supp_save(fig, name):
    png = SUP_FIG / f"{name}.png"
    pdf = SUP_FIG / f"{name}.pdf"

    fig.savefig(
        png,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf,
        bbox_inches="tight",
        facecolor="white"
    )

    print("Saved:", png.name)
    return png, pdf


def clean_map_axis(ax):
    ax.set_axis_off()
    ax.set_aspect("equal")


def label_panel(ax, letter):
    ax.text(
        0.01,
        0.99,
        letter,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=10,
        fontweight="bold"
    )


def geometry_for_case(case):
    if case == "SIMULATION":
        G = SIM_HEX[["HEX_ID", "geometry"]].copy()
    else:
        G = HEX[["HEX_ID", "geometry"]].copy()

    G["HEX_ID"] = G["HEX_ID"].astype(str)

    G = (
        G
        .drop_duplicates("HEX_ID")
        .loc[G.geometry.notna()]
        .copy()
    )

    G = G.loc[~G.geometry.is_empty].copy()

    if G.crs is None:
        G = G.set_crs("EPSG:5070")
    elif G.crs.to_epsg() != 5070:
        G = G.to_crs(5070)

    return G


def plot_gv_map(ax, direct, geom, time_col, time_value):
    d = direct.loc[
        pd.to_numeric(
            direct[time_col],
            errors="coerce"
        ).eq(time_value),
        ["HEX_ID", "GV_CELL"]
    ].copy()

    d["HEX_ID"] = d["HEX_ID"].astype(str)

    m = geom.merge(
        d,
        on="HEX_ID",
        how="left",
        validate="1:1"
    )

    missing = m.loc[m["GV_CELL"].isna()]

    if len(missing):
        missing.plot(
            ax=ax,
            facecolor="#f1f1f1",
            edgecolor="none"
        )

    for cell in range(1, 10):
        q = m.loc[
            pd.to_numeric(
                m["GV_CELL"],
                errors="coerce"
            ).eq(cell)
        ]

        if len(q):
            q.plot(
                ax=ax,
                facecolor=GV[cell],
                edgecolor="none"
            )

    clean_map_axis(ax)


def draw_gv_matrix(ax, title="GVSTMR matrix"):
    ax.set_xlim(0, 3)
    ax.set_ylim(0, 3)

    for s in range(3):
        for t in range(3):
            cell = s * 3 + t + 1

            ax.add_patch(
                Rectangle(
                    (t, s),
                    1,
                    1,
                    facecolor=GV[cell],
                    edgecolor="white",
                    linewidth=1.0
                )
            )

    ax.set_xticks(
        [0.5, 1.5, 2.5],
        ["Low", "Medium", "High"]
    )

    ax.set_yticks(
        [0.5, 1.5, 2.5],
        ["Low", "Medium", "High"]
    )

    ax.set_xlabel(r"Temporal relationship $\rho^T$")
    ax.set_ylabel(r"Spatial relationship $\rho^S$")
    ax.set_title(title, fontweight="bold")

    ax.tick_params(length=0)

    for sp in ax.spines.values():
        sp.set_visible(False)

    ax.set_aspect("equal")


def model_metric_table(df, model, case):
    rows = []

    d = df.copy()

    d["TIME"] = pd.to_numeric(
        d["TIME"],
        errors="coerce"
    )

    d["MODEL_VALUE"] = pd.to_numeric(
        d["MODEL_VALUE"],
        errors="coerce"
    )

    d["OBS_VALUE"] = pd.to_numeric(
        d["OBS_VALUE"],
        errors="coerce"
    )

    for tt, q in d.groupby("TIME"):

        y = q["OBS_VALUE"].to_numpy(float)
        p = q["MODEL_VALUE"].to_numpy(float)

        ok = np.isfinite(y) & np.isfinite(p)
        y = y[ok]
        p = p[ok]

        if len(y) < 3:
            continue

        rows.append({
            "MODEL": model,
            "CASE": case,
            "TIME": tt,
            "N": len(y),
            "R2": r2_score(y, p),
            "RMSE": mean_squared_error(y, p) ** 0.5,
            "MAE": mean_absolute_error(y, p),
            "BIAS": float(np.mean(p - y)),
            "PEARSON_R": float(np.corrcoef(y, p)[0, 1]),
        })

    return pd.DataFrame(rows)


def detect_uncertainty_col(df):
    preferred = [
        "GGPR_POSTERIOR_SD_C",
        "GGPR_POSTERIOR_SD",
        "POSTERIOR_SD_C",
        "POSTERIOR_SD",
        "PRED_SD_C",
        "PREDICTIVE_SD_C",
        "UNCERTAINTY_C",
        "UNCERTAINTY",
    ]

    for c in preferred:
        if c in df.columns:
            return c

    for c in df.columns:
        u = c.upper()

        if (
            "UNCERTAINT" in u
            or
            (
                "POSTERIOR" in u
                and
                ("SD" in u or "STD" in u)
            )
        ):
            if pd.api.types.is_numeric_dtype(df[c]):
                return c

    return None


def environmental_geoshap_cols(df):
    cols = []

    for c in df.columns:
        u = c.upper()

        if not u.startswith("GEOSHAP_"):
            continue

        if any(
            token in u
            for token in [
                "BASE",
                "SUM",
                "RECON",
                "ERROR",
                "RESID",
                "INTERACTION",
            ]
        ):
            continue

        base = re.sub(
            r"^GEOSHAP_",
            "",
            c,
            flags=re.I
        )

        norm = re.sub(
            r"[^A-Z0-9]",
            "",
            base.upper()
        )

        # Keep environmental features; separate GEO/location from the feature ranking.
        if norm in ["GEO", "GEOC", "LOCATION", "SPATIAL"]:
            continue

        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)

    return cols


def short_feature_name(col):
    x = re.sub(
        r"^GEOSHAP_",
        "",
        str(col),
        flags=re.I
    )
    x = re.sub(r"_C$", "", x)
    return x


MANIFEST = []

def register(fig_no, title, status="created"):
    MANIFEST.append({
        "FIGURE": fig_no,
        "TITLE": title,
        "STATUS": status,
    })


print("=" * 105)
print("GVSTMR SUPPLEMENTARY CARTOGRAPHY")
print("=" * 105)
print("Output:", SUP_ROOT)
print("No MGWR / GTWR / GGPR model refitting will be performed.")


In [ ]:
# ================================================================================================
# FIGURES S1–S2 — COMPLETE 10-TIME DIRECT-GVSTMR ATLASES
# ================================================================================================

def full_atlas(
    direct,
    geom,
    time_col,
    times,
    title,
    out_name
):
    fig = plt.figure(
        figsize=(15.2, 6.2),
        facecolor="white"
    )

    gs = fig.add_gridspec(
        2,
        6,
        width_ratios=[1, 1, 1, 1, 1, 0.72],
        left=0.025,
        right=0.985,
        top=0.88,
        bottom=0.06,
        wspace=0.03,
        hspace=0.08
    )

    for i, tt in enumerate(times):
        r = i // 5
        c = i % 5

        ax = fig.add_subplot(
            gs[r, c]
        )

        plot_gv_map(
            ax,
            direct,
            geom,
            time_col,
            tt
        )

        ax.set_title(
            str(tt),
            fontsize=9.5,
            fontweight="bold"
        )

    leg = fig.add_subplot(
        gs[:, 5]
    )

    draw_gv_matrix(
        leg
    )

    fig.suptitle(
        title,
        fontsize=12,
        fontweight="bold"
    )

    supp_save(
        fig,
        out_name
    )

    plt.show()
    plt.close(fig)


SIM_GEOM_SUPP = geometry_for_case(
    "SIMULATION"
)

REAL_GEOM_SUPP = geometry_for_case(
    "REAL"
)

full_atlas(
    SIM_DIRECT,
    SIM_GEOM_SUPP,
    "TIME",
    list(SIM_TIMES),
    "Figure S1. Complete GVSTMR evolution in the CONUS simulation",
    "Figure_S01_Simulation_GVSTMR_All_Times"
)

register(
    "S1",
    "Complete Simulation GVSTMR atlas, T1–T10"
)

full_atlas(
    REAL_DIRECT,
    REAL_GEOM_SUPP,
    "YEAR",
    list(YEARS),
    "Figure S2. Complete real-case GVSTMR evolution for NDVI–JJA LST",
    "Figure_S02_Real_GVSTMR_All_Years"
)

register(
    "S2",
    "Complete Real-case GVSTMR atlas, 2015–2024"
)


In [ ]:
# ================================================================================================
# FIGURE S3 — CORE-GVSTMR PARAMETER SENSITIVITY
#
# Baseline from the manuscript:
#     K = 30
#     Gaussian temporal bandwidth h = 2
#
# Sensitivity grid:
#     K = 15, 30, 60
#     h = 1, 2, 3
#
# Outputs:
#     Spearman agreement of rhoS with baseline
#     Spearman agreement of rhoT with baseline
#     Exact nine-state GVSTMR-class agreement with baseline
#
# No model fitting.
# ================================================================================================

K_VALUES = [
    15,
    30,
    60
]

H_VALUES = [
    1.0,
    2.0,
    3.0
]


def _row_corr_local(A, B):
    valid = np.isfinite(A) & np.isfinite(B)

    n = valid.sum(
        axis=1
    )

    AA = np.where(
        valid,
        A,
        0.0
    )

    BB = np.where(
        valid,
        B,
        0.0
    )

    mean_a = np.divide(
        AA.sum(axis=1),
        n,
        out=np.full(len(n), np.nan),
        where=n > 0
    )

    mean_b = np.divide(
        BB.sum(axis=1),
        n,
        out=np.full(len(n), np.nan),
        where=n > 0
    )

    da = np.where(
        valid,
        A - mean_a[:, None],
        0.0
    )

    db = np.where(
        valid,
        B - mean_b[:, None],
        0.0
    )

    num = (
        da * db
    ).sum(
        axis=1
    )

    den = np.sqrt(
        (da * da).sum(axis=1)
        *
        (db * db).sum(axis=1)
    )

    return np.divide(
        num,
        den,
        out=np.full(len(n), np.nan),
        where=(
            (n >= 3)
            &
            np.isfinite(den)
            &
            (den > 1e-12)
        )
    )


def _weighted_temporal_corr_local(X, Y, h):
    n_time, n_hex = X.shape

    out = np.full(
        (n_time, n_hex),
        np.nan,
        dtype=float
    )

    tt = np.arange(
        n_time,
        dtype=float
    )

    for center in range(n_time):

        w = np.exp(
            -0.5
            *
            (
                (tt - center)
                /
                float(h)
            ) ** 2
        )

        w = (
            w
            /
            w.sum()
        )

        valid = np.isfinite(X) & np.isfinite(Y)

        W = (
            w[:, None]
            *
            valid
        )

        sw = W.sum(
            axis=0
        )

        mx = np.divide(
            (W * np.where(valid, X, 0.0)).sum(axis=0),
            sw,
            out=np.full(n_hex, np.nan),
            where=sw > 0
        )

        my = np.divide(
            (W * np.where(valid, Y, 0.0)).sum(axis=0),
            sw,
            out=np.full(n_hex, np.nan),
            where=sw > 0
        )

        dx = np.where(
            valid,
            X - mx[None, :],
            0.0
        )

        dy = np.where(
            valid,
            Y - my[None, :],
            0.0
        )

        vx = (
            W * dx ** 2
        ).sum(
            axis=0
        )

        vy = (
            W * dy ** 2
        ).sum(
            axis=0
        )

        cv = (
            W * dx * dy
        ).sum(
            axis=0
        )

        den = np.sqrt(
            vx * vy
        )

        out[
            center
        ] = np.divide(
            cv,
            den,
            out=np.full(n_hex, np.nan),
            where=den > 1e-12
        )

    return np.clip(
        out,
        -1,
        1
    )


def prepare_sensitivity_case(
    panel,
    geom,
    x_col,
    y_col,
    time_col,
    times
):
    P = panel.copy()

    P["HEX_ID"] = (
        P["HEX_ID"]
        .astype(str)
    )

    G = geom[
        [
            "HEX_ID",
            "geometry"
        ]
    ].copy()

    G["HEX_ID"] = (
        G["HEX_ID"]
        .astype(str)
    )

    if G.crs is None:
        G = G.set_crs("EPSG:5070")
    elif G.crs.to_epsg() != 5070:
        G = G.to_crs(5070)

    G = (
        G
        .drop_duplicates("HEX_ID")
        .copy()
    )

    cent = G.geometry.centroid

    G["X_KM"] = (
        cent.x
        /
        1000.0
    )

    G["Y_KM"] = (
        cent.y
        /
        1000.0
    )

    P = P.loc[
        P[time_col].isin(times)
    ].copy()

    complete = (
        P.groupby("HEX_ID")
        .agg(
            NT=(time_col, "nunique"),
            NX=(x_col, lambda s: int(s.notna().sum())),
            NY=(y_col, lambda s: int(s.notna().sum())),
        )
    )

    ids = complete.index[
        complete["NT"].eq(len(times))
        &
        complete["NX"].eq(len(times))
        &
        complete["NY"].eq(len(times))
    ].astype(str)

    order = (
        G.loc[
            G["HEX_ID"].isin(ids),
            "HEX_ID"
        ]
        .astype(str)
        .tolist()
    )

    X = (
        P.loc[
            P["HEX_ID"].isin(order)
        ]
        .pivot(
            index=time_col,
            columns="HEX_ID",
            values=x_col
        )
        .reindex(
            index=times,
            columns=order
        )
        .to_numpy(float)
    )

    Y = (
        P.loc[
            P["HEX_ID"].isin(order)
        ]
        .pivot(
            index=time_col,
            columns="HEX_ID",
            values=y_col
        )
        .reindex(
            index=times,
            columns=order
        )
        .to_numpy(float)
    )

    coords = (
        G
        .set_index("HEX_ID")
        .loc[
            order,
            ["X_KM", "Y_KM"]
        ]
        .to_numpy(float)
    )

    max_k = max(
        K_VALUES
    )

    _, nbr = cKDTree(
        coords
    ).query(
        coords,
        k=min(
            max_k + 1,
            len(coords)
        )
    )

    if nbr.ndim == 1:
        nbr = nbr[:, None]

    return X, Y, nbr


def score_sensitivity(
    X,
    Y,
    nbr,
    k,
    h
):
    use_nbr = nbr[
        :,
        :min(
            k + 1,
            nbr.shape[1]
        )
    ]

    rho_s = np.empty_like(
        X,
        dtype=float
    )

    for ti in range(
        X.shape[0]
    ):
        rho_s[
            ti
        ] = _row_corr_local(
            X[ti][use_nbr],
            Y[ti][use_nbr]
        )

    rho_t = _weighted_temporal_corr_local(
        X,
        Y,
        h
    )

    s_valid = rho_s[
        np.isfinite(rho_s)
    ]

    t_valid = rho_t[
        np.isfinite(rho_t)
    ]

    s_lo, s_hi = np.quantile(
        s_valid,
        [1 / 3, 2 / 3]
    )

    t_lo, t_hi = np.quantile(
        t_valid,
        [1 / 3, 2 / 3]
    )

    s_cls = np.where(
        rho_s <= s_lo,
        1,
        np.where(
            rho_s <= s_hi,
            2,
            3
        )
    )

    t_cls = np.where(
        rho_t <= t_lo,
        1,
        np.where(
            rho_t <= t_hi,
            2,
            3
        )
    )

    gv = (
        (s_cls - 1)
        *
        3
        +
        t_cls
    )

    return rho_s, rho_t, gv


def sensitivity_table(
    case,
    panel,
    geom,
    x_col,
    y_col,
    time_col,
    times
):
    X, Y, nbr = prepare_sensitivity_case(
        panel,
        geom,
        x_col,
        y_col,
        time_col,
        times
    )

    bs, bt, bg = score_sensitivity(
        X,
        Y,
        nbr,
        30,
        2.0
    )

    rows = []

    for k in K_VALUES:
        for h in H_VALUES:
            rs, rt, gv = score_sensitivity(
                X,
                Y,
                nbr,
                k,
                h
            )

            ok_s = np.isfinite(bs) & np.isfinite(rs)
            ok_t = np.isfinite(bt) & np.isfinite(rt)
            ok_g = np.isfinite(bg) & np.isfinite(gv)

            rows.append({
                "CASE": case,
                "K": k,
                "H": h,
                "RHO_S_SPEARMAN_TO_BASELINE":
                    float(
                        spearmanr(
                            bs[ok_s],
                            rs[ok_s]
                        ).statistic
                    ),
                "RHO_T_SPEARMAN_TO_BASELINE":
                    float(
                        spearmanr(
                            bt[ok_t],
                            rt[ok_t]
                        ).statistic
                    ),
                "GV_CELL_EXACT_AGREEMENT":
                    float(
                        np.mean(
                            bg[ok_g]
                            ==
                            gv[ok_g]
                        )
                    ),
            })

    return pd.DataFrame(
        rows
    )


SENS_SIM = sensitivity_table(
    "SIMULATION",
    SIM_PANEL,
    SIM_HEX,
    "X",
    "Y",
    "TIME",
    list(SIM_TIMES)
)

SENS_REAL = sensitivity_table(
    "REAL",
    PANEL,
    HEX,
    "NDVI",
    "LST_MEAN_C",
    "YEAR",
    list(YEARS)
)

SENS = pd.concat(
    [
        SENS_SIM,
        SENS_REAL
    ],
    ignore_index=True
)

SENS.to_csv(
    SUP_DATA /
    "TABLE_S03_GVSTMR_PARAMETER_SENSITIVITY.csv",
    index=False
)


def heatmap_sensitivity(
    ax,
    d,
    value_col,
    title
):
    pivot = (
        d.pivot(
            index="K",
            columns="H",
            values=value_col
        )
        .reindex(
            index=K_VALUES,
            columns=H_VALUES
        )
    )

    im = ax.imshow(
        pivot.to_numpy(float),
        vmin=0,
        vmax=1,
        cmap=LinearSegmentedColormap.from_list(
            "agreement",
            [
                "#f7f7f7",
                GV[2],
                GV[5],
                GV[8],
                GV[9]
            ]
        ),
        aspect="auto"
    )

    ax.set_xticks(
        range(len(H_VALUES)),
        [f"{x:g}" for x in H_VALUES]
    )

    ax.set_yticks(
        range(len(K_VALUES)),
        [str(x) for x in K_VALUES]
    )

    ax.set_xlabel(
        "Temporal bandwidth h"
    )

    ax.set_ylabel(
        "Spatial neighbors K"
    )

    ax.set_title(
        title,
        fontweight="bold"
    )

    values = pivot.to_numpy(float)

    for r in range(values.shape[0]):
        for c in range(values.shape[1]):
            ax.text(
                c,
                r,
                f"{values[r, c]:.2f}",
                ha="center",
                va="center",
                fontsize=7.4,
                color=(
                    "white"
                    if values[r, c] > 0.72
                    else "black"
                )
            )

    return im


fig, axes = plt.subplots(
    2,
    3,
    figsize=(12.5, 6.7),
    constrained_layout=True
)

metrics = [
    (
        "RHO_S_SPEARMAN_TO_BASELINE",
        r"Spatial-score agreement ($\rho^S$)"
    ),
    (
        "RHO_T_SPEARMAN_TO_BASELINE",
        r"Temporal-score agreement ($\rho^T$)"
    ),
    (
        "GV_CELL_EXACT_AGREEMENT",
        "Exact 9-state GVSTMR agreement"
    )
]

for row, (
    case,
    d
) in enumerate(
    [
        ("Simulation", SENS_SIM),
        ("Real case", SENS_REAL)
    ]
):
    for col, (
        metric,
        title
    ) in enumerate(metrics):
        heatmap_sensitivity(
            axes[row, col],
            d,
            metric,
            title
        )

        if col == 0:
            axes[row, col].text(
                -0.33,
                0.5,
                case,
                transform=axes[row, col].transAxes,
                rotation=90,
                va="center",
                ha="center",
                fontsize=10,
                fontweight="bold"
            )

fig.suptitle(
    "Figure S3. Robustness of the core GVSTMR operator to neighborhood and temporal-bandwidth choices\n"
    "Values are agreement with the manuscript baseline K=30 and h=2",
    fontsize=11.5,
    fontweight="bold"
)

supp_save(
    fig,
    "Figure_S03_GVSTMR_Parameter_Sensitivity"
)

plt.show()
plt.close(fig)

register(
    "S3",
    "Core-GVSTMR sensitivity to spatial K and temporal bandwidth h"
)


In [ ]:
# ================================================================================================
# FIGURE S4 — NINE-STATE TRANSITIONS + PERSISTENCE
# ================================================================================================

def transition_summary(
    direct,
    time_col,
    times,
    case
):
    d = direct[
        [
            "HEX_ID",
            time_col,
            "GV_CELL"
        ]
    ].copy()

    d["HEX_ID"] = (
        d["HEX_ID"]
        .astype(str)
    )

    d[time_col] = pd.to_numeric(
        d[time_col],
        errors="coerce"
    )

    d["GV_CELL"] = pd.to_numeric(
        d["GV_CELL"],
        errors="coerce"
    )

    wide = (
        d.pivot(
            index="HEX_ID",
            columns=time_col,
            values="GV_CELL"
        )
        .reindex(
            columns=times
        )
    )

    count = np.zeros(
        (9, 9),
        dtype=float
    )

    persistence_rows = []

    for a, b in zip(
        times[:-1],
        times[1:]
    ):
        x = wide[a].to_numpy(float)
        y = wide[b].to_numpy(float)

        ok = (
            np.isfinite(x)
            &
            np.isfinite(y)
        )

        xx = x[ok].astype(int)
        yy = y[ok].astype(int)

        for s0, s1 in zip(
            xx,
            yy
        ):
            if 1 <= s0 <= 9 and 1 <= s1 <= 9:
                count[
                    s0 - 1,
                    s1 - 1
                ] += 1

        persistence_rows.append({
            "CASE": case,
            "FROM_TIME": a,
            "TO_TIME": b,
            "PERSISTENCE":
                float(
                    np.mean(
                        xx == yy
                    )
                )
                if len(xx)
                else np.nan,
            "N": len(xx),
        })

    row_sum = count.sum(
        axis=1,
        keepdims=True
    )

    prob = np.divide(
        count,
        row_sum,
        out=np.zeros_like(count),
        where=row_sum > 0
    )

    return (
        prob,
        pd.DataFrame(
            persistence_rows
        )
    )


SIM_TRANS, SIM_PERSIST = transition_summary(
    SIM_DIRECT,
    "TIME",
    list(SIM_TIMES),
    "SIMULATION"
)

REAL_TRANS, REAL_PERSIST = transition_summary(
    REAL_DIRECT,
    "YEAR",
    list(YEARS),
    "REAL"
)

PERSIST = pd.concat(
    [
        SIM_PERSIST,
        REAL_PERSIST
    ],
    ignore_index=True
)

PERSIST.to_csv(
    SUP_DATA /
    "TABLE_S04_GVSTMR_TEMPORAL_PERSISTENCE.csv",
    index=False
)

pd.DataFrame(
    SIM_TRANS,
    index=range(1, 10),
    columns=range(1, 10)
).to_csv(
    SUP_DATA /
    "TABLE_S04_SIMULATION_TRANSITION_MATRIX.csv"
)

pd.DataFrame(
    REAL_TRANS,
    index=range(1, 10),
    columns=range(1, 10)
).to_csv(
    SUP_DATA /
    "TABLE_S04_REAL_TRANSITION_MATRIX.csv"
)


def transition_heatmap(
    ax,
    matrix,
    title
):
    im = ax.imshow(
        matrix,
        vmin=0,
        vmax=max(
            0.01,
            float(
                np.nanpercentile(
                    matrix,
                    98
                )
            )
        ),
        cmap=UNC_CMAP,
        aspect="equal"
    )

    ax.set_xticks(
        range(9),
        range(1, 10)
    )

    ax.set_yticks(
        range(9),
        range(1, 10)
    )

    ax.set_xlabel(
        "GVSTMR state at t+1"
    )

    ax.set_ylabel(
        "GVSTMR state at t"
    )

    ax.set_title(
        title,
        fontweight="bold"
    )

    for r in range(9):
        for c in range(9):
            v = matrix[r, c]

            if v >= 0.05:
                ax.text(
                    c,
                    r,
                    f"{100*v:.0f}",
                    ha="center",
                    va="center",
                    fontsize=6.1,
                    color=(
                        "white"
                        if v > 0.28
                        else "black"
                    )
                )

    return im


fig = plt.figure(
    figsize=(12.2, 7.2),
    facecolor="white"
)

gs = fig.add_gridspec(
    2,
    2,
    height_ratios=[1.25, 0.75],
    left=0.07,
    right=0.98,
    top=0.88,
    bottom=0.10,
    wspace=0.18,
    hspace=0.30
)

ax_a = fig.add_subplot(
    gs[0, 0]
)

ax_b = fig.add_subplot(
    gs[0, 1]
)

transition_heatmap(
    ax_a,
    SIM_TRANS,
    "Simulation: row-normalized state transitions (%)"
)

transition_heatmap(
    ax_b,
    REAL_TRANS,
    "Real case: row-normalized state transitions (%)"
)

ax_c = fig.add_subplot(
    gs[1, 0]
)

ax_d = fig.add_subplot(
    gs[1, 1]
)

for ax, d, title in [
    (
        ax_c,
        SIM_PERSIST,
        "Simulation"
    ),
    (
        ax_d,
        REAL_PERSIST,
        "Real case"
    )
]:
    x = np.arange(
        len(d)
    )

    ax.plot(
        x,
        d["PERSISTENCE"],
        marker="o",
        lw=1.6,
        color=GV[8]
    )

    labels = [
        f"{int(a)}→{int(b)}"
        for a, b in zip(
            d["FROM_TIME"],
            d["TO_TIME"]
        )
    ]

    ax.set_xticks(
        x,
        labels,
        rotation=45,
        ha="right"
    )

    ax.set_ylim(
        0,
        1
    )

    ax.set_ylabel(
        "Exact-state persistence"
    )

    ax.set_title(
        f"{title}: consecutive-time persistence",
        fontweight="bold"
    )

    ax.grid(
        axis="y",
        alpha=0.20
    )

fig.suptitle(
    "Figure S4. Transition structure and temporal persistence of the nine GVSTMR relationship states",
    fontsize=11.5,
    fontweight="bold"
)

supp_save(
    fig,
    "Figure_S04_GVSTMR_Transitions_and_Persistence"
)

plt.show()
plt.close(fig)

register(
    "S4",
    "Nine-state GVSTMR transitions and temporal persistence"
)


In [ ]:
# ================================================================================================
# FIGURE S5 — TIME-RESOLVED MODEL PERFORMANCE
# FIGURE S6 — CROSS-MODEL GVSTMR AGREEMENT
# ================================================================================================

MODEL_OBJECTS_AVAILABLE = all(
    name in globals()
    for name in [
        "MGWR_SIM",
        "MGWR_REAL",
        "GTWR_SIM",
        "GTWR_REAL",
        "GGPR_SIM",
        "GGPR_REAL",
    ]
)

if not MODEL_OBJECTS_AVAILABLE:

    print(
        "S5/S6 skipped: run the completed MGWR, GTWR and GGPR model-GVSTMR cells "
        "from the main notebook first."
    )

    register(
        "S5",
        "Time-resolved predictive performance",
        status="skipped — model objects missing"
    )

    register(
        "S6",
        "Cross-model GVSTMR agreement",
        status="skipped — model objects missing"
    )

else:

    MODEL_CASES = {
        "SIMULATION": {
            "MGWR": MGWR_SIM,
            "GTWR": GTWR_SIM,
            "GGPR": GGPR_SIM,
        },
        "REAL": {
            "MGWR": MGWR_REAL,
            "GTWR": GTWR_REAL,
            "GGPR": GGPR_REAL,
        }
    }

    # --------------------------------------------------------------------------------------------
    # S5
    # --------------------------------------------------------------------------------------------

    metric_parts = []

    for case, models in MODEL_CASES.items():
        for model, d in models.items():
            metric_parts.append(
                model_metric_table(
                    d,
                    model,
                    case
                )
            )

    TIME_METRICS = pd.concat(
        metric_parts,
        ignore_index=True
    )

    TIME_METRICS.to_csv(
        SUP_DATA /
        "TABLE_S05_MODEL_METRICS_BY_TIME.csv",
        index=False
    )

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(14.0, 6.6),
        constrained_layout=True
    )

    metric_specs = [
        ("R2", r"$R^2$"),
        ("RMSE", "RMSE"),
        ("MAE", "MAE"),
        ("BIAS", "Bias"),
    ]

    for row, case in enumerate(
        ["SIMULATION", "REAL"]
    ):
        dcase = TIME_METRICS.loc[
            TIME_METRICS["CASE"].eq(case)
        ]

        for col, (
            metric,
            ylabel
        ) in enumerate(
            metric_specs
        ):
            ax = axes[
                row,
                col
            ]

            for model in [
                "MGWR",
                "GTWR",
                "GGPR"
            ]:
                q = (
                    dcase.loc[
                        dcase["MODEL"].eq(model)
                    ]
                    .sort_values("TIME")
                )

                ax.plot(
                    q["TIME"],
                    q[metric],
                    marker="o",
                    ms=3.8,
                    lw=1.5,
                    color=MODEL_COLORS[model],
                    label=model
                )

            if metric == "BIAS":
                ax.axhline(
                    0,
                    color="#777777",
                    lw=0.8,
                    ls="--"
                )

            ax.set_title(
                ylabel,
                fontweight="bold"
            )

            ax.set_xlabel(
                "Time"
                if case == "SIMULATION"
                else "Year"
            )

            ax.grid(
                axis="y",
                alpha=0.18
            )

            if col == 0:
                ax.set_ylabel(
                    "Simulation"
                    if case == "SIMULATION"
                    else "Real case"
                )

            if row == 0 and col == 3:
                ax.legend(
                    frameon=False,
                    loc="best"
                )

    fig.suptitle(
        "Figure S5. Time-resolved predictive performance of the three supporting models",
        fontsize=11.5,
        fontweight="bold"
    )

    supp_save(
        fig,
        "Figure_S05_Model_Performance_Through_Time"
    )

    plt.show()
    plt.close(fig)

    register(
        "S5",
        "Time-resolved predictive performance of MGWR, GTWR and GGPR"
    )

    # --------------------------------------------------------------------------------------------
    # S6
    # --------------------------------------------------------------------------------------------

    AGREEMENT_ROWS = []

    for case, models in MODEL_CASES.items():

        names = [
            "MGWR",
            "GTWR",
            "GGPR"
        ]

        for i, a in enumerate(names):
            for b in names[i:]:

                A = models[a][
                    [
                        "HEX_ID",
                        "TIME",
                        "GV_CELL",
                        "RHO_S",
                        "RHO_T"
                    ]
                ].copy()

                B = models[b][
                    [
                        "HEX_ID",
                        "TIME",
                        "GV_CELL",
                        "RHO_S",
                        "RHO_T"
                    ]
                ].copy()

                A["HEX_ID"] = A["HEX_ID"].astype(str)
                B["HEX_ID"] = B["HEX_ID"].astype(str)

                M = A.merge(
                    B,
                    on=[
                        "HEX_ID",
                        "TIME"
                    ],
                    how="inner",
                    suffixes=("_A", "_B"),
                    validate="1:1"
                )

                gva = pd.to_numeric(
                    M["GV_CELL_A"],
                    errors="coerce"
                ).to_numpy(float)

                gvb = pd.to_numeric(
                    M["GV_CELL_B"],
                    errors="coerce"
                ).to_numpy(float)

                rsa = pd.to_numeric(
                    M["RHO_S_A"],
                    errors="coerce"
                ).to_numpy(float)

                rsb = pd.to_numeric(
                    M["RHO_S_B"],
                    errors="coerce"
                ).to_numpy(float)

                rta = pd.to_numeric(
                    M["RHO_T_A"],
                    errors="coerce"
                ).to_numpy(float)

                rtb = pd.to_numeric(
                    M["RHO_T_B"],
                    errors="coerce"
                ).to_numpy(float)

                okg = np.isfinite(gva) & np.isfinite(gvb)
                oks = np.isfinite(rsa) & np.isfinite(rsb)
                okt = np.isfinite(rta) & np.isfinite(rtb)

                AGREEMENT_ROWS.append({
                    "CASE": case,
                    "MODEL_A": a,
                    "MODEL_B": b,
                    "N_COMMON": len(M),
                    "GV_CELL_EXACT_AGREEMENT":
                        float(
                            np.mean(
                                gva[okg] == gvb[okg]
                            )
                        )
                        if okg.any()
                        else np.nan,
                    "RHO_S_SPEARMAN":
                        float(
                            spearmanr(
                                rsa[oks],
                                rsb[oks]
                            ).statistic
                        )
                        if oks.sum() >= 3
                        else np.nan,
                    "RHO_T_SPEARMAN":
                        float(
                            spearmanr(
                                rta[okt],
                                rtb[okt]
                            ).statistic
                        )
                        if okt.sum() >= 3
                        else np.nan,
                })

    AGREEMENT = pd.DataFrame(
        AGREEMENT_ROWS
    )

    # Symmetrize for plotting
    rev = AGREEMENT.loc[
        AGREEMENT["MODEL_A"] != AGREEMENT["MODEL_B"]
    ].rename(
        columns={
            "MODEL_A": "MODEL_B",
            "MODEL_B": "MODEL_A"
        }
    )

    AGREEMENT_FULL = pd.concat(
        [
            AGREEMENT,
            rev
        ],
        ignore_index=True
    )

    AGREEMENT_FULL.to_csv(
        SUP_DATA /
        "TABLE_S06_CROSS_MODEL_GVSTMR_AGREEMENT.csv",
        index=False
    )


    def model_matrix(
        d,
        metric
    ):
        models = [
            "MGWR",
            "GTWR",
            "GGPR"
        ]

        M = np.full(
            (3, 3),
            np.nan
        )

        for i, a in enumerate(models):
            for j, b in enumerate(models):
                q = d.loc[
                    d["MODEL_A"].eq(a)
                    &
                    d["MODEL_B"].eq(b)
                ]

                if len(q):
                    M[i, j] = q.iloc[0][metric]

        return M


    fig, axes = plt.subplots(
        2,
        3,
        figsize=(10.8, 7.0),
        constrained_layout=True
    )

    specs = [
        (
            "GV_CELL_EXACT_AGREEMENT",
            "Exact GVSTMR-state agreement"
        ),
        (
            "RHO_S_SPEARMAN",
            r"Spatial-score agreement ($\rho^S$)"
        ),
        (
            "RHO_T_SPEARMAN",
            r"Temporal-score agreement ($\rho^T$)"
        )
    ]

    for row, case in enumerate(
        ["SIMULATION", "REAL"]
    ):
        dc = AGREEMENT_FULL.loc[
            AGREEMENT_FULL["CASE"].eq(case)
        ]

        for col, (
            metric,
            title
        ) in enumerate(specs):

            M = model_matrix(
                dc,
                metric
            )

            ax = axes[
                row,
                col
            ]

            ax.imshow(
                M,
                vmin=0,
                vmax=1,
                cmap=UNC_CMAP
            )

            ax.set_xticks(
                range(3),
                [
                    "MGWR",
                    "GTWR",
                    "GGPR"
                ]
            )

            ax.set_yticks(
                range(3),
                [
                    "MGWR",
                    "GTWR",
                    "GGPR"
                ]
            )

            ax.set_title(
                title,
                fontweight="bold"
            )

            for i in range(3):
                for j in range(3):
                    if np.isfinite(M[i, j]):
                        ax.text(
                            j,
                            i,
                            f"{M[i, j]:.2f}",
                            ha="center",
                            va="center",
                            fontsize=7.5,
                            color=(
                                "white"
                                if M[i, j] > 0.72
                                else "black"
                            )
                        )

            if col == 0:
                ax.set_ylabel(
                    "Simulation"
                    if case == "SIMULATION"
                    else "Real case",
                    fontweight="bold"
                )

    fig.suptitle(
        "Figure S6. Cross-model agreement under the common GVSTMR relationship grammar",
        fontsize=11.5,
        fontweight="bold"
    )

    supp_save(
        fig,
        "Figure_S06_Cross_Model_GVSTMR_Agreement"
    )

    plt.show()
    plt.close(fig)

    register(
        "S6",
        "Cross-model GVSTMR, spatial-score and temporal-score agreement"
    )


In [ ]:
# ================================================================================================
# FIGURE S7 — FULL MGWR LOCAL-COEFFICIENT ATLAS + BANDWIDTHS
# ================================================================================================

def _load_raw_mgwr_coefficients():
    if "real_mgwr" in globals():
        return real_mgwr.copy()

    if "REAL_MGWR_PATH" in globals():
        p = Path(REAL_MGWR_PATH)

        if p.exists():
            if p.suffix.lower() == ".parquet":
                return pd.read_parquet(p)

            return pd.read_csv(p)

    # conservative fallback
    candidates = []

    for p in Path("/content").rglob("*"):
        if not p.is_file():
            continue

        name = p.name.upper()

        if (
            "MGWR" in name
            and
            "LOCAL" in name
            and
            p.suffix.lower() in [".parquet", ".csv"]
        ):
            candidates.append(p)

    for p in candidates:
        try:
            d = (
                pd.read_parquet(p)
                if p.suffix.lower() == ".parquet"
                else pd.read_csv(p)
            )

            if "HEX_ID" in d.columns:
                return d
        except Exception:
            continue

    return None


def _beta_col(df, feature):
    if "resolve_beta_column" in globals():
        try:
            return resolve_beta_column(
                df,
                feature
            )
        except Exception:
            pass

    for c in [
        f"BETA_{feature}",
        f"beta_{feature}",
        feature
    ]:
        if c in df.columns:
            return c

    matches = [
        c
        for c in df.columns
        if (
            "beta" in c.lower()
            and
            feature.lower() in c.lower()
        )
    ]

    return (
        matches[0]
        if matches
        else None
    )


RAW_MGWR = _load_raw_mgwr_coefficients()

if RAW_MGWR is None:

    print(
        "S7 skipped: full MGWR local-coefficient table was not found."
    )

    register(
        "S7",
        "Full MGWR coefficient atlas",
        status="skipped — coefficient table missing"
    )

else:

    RAW_MGWR["HEX_ID"] = (
        RAW_MGWR["HEX_ID"]
        .astype(str)
    )

    features_here = [
        f
        for f in FEATURES
        if _beta_col(
            RAW_MGWR,
            f
        ) is not None
    ]

    beta_map = {
        f: _beta_col(
            RAW_MGWR,
            f
        )
        for f in features_here
    }

    coeff_values = []

    for f, c in beta_map.items():
        v = pd.to_numeric(
            RAW_MGWR[c],
            errors="coerce"
        ).to_numpy(float)

        coeff_values.extend(
            np.abs(
                v[
                    np.isfinite(v)
                ]
            ).tolist()
        )

    vmax = float(
        np.nanpercentile(
            coeff_values,
            99
        )
    )

    vmax = max(
        vmax,
        1e-6
    )

    norm = TwoSlopeNorm(
        vmin=-vmax,
        vcenter=0,
        vmax=vmax
    )

    G = geometry_for_case(
        "REAL"
    )

    fig = plt.figure(
        figsize=(14.3, 10.0),
        facecolor="white"
    )

    gs = fig.add_gridspec(
        3,
        3,
        left=0.035,
        right=0.96,
        top=0.90,
        bottom=0.10,
        wspace=0.06,
        hspace=0.12
    )

    for idx, feature in enumerate(
        features_here[:8]
    ):

        ax = fig.add_subplot(
            gs[
                idx // 3,
                idx % 3
            ]
        )

        col = beta_map[
            feature
        ]

        q = RAW_MGWR[
            [
                "HEX_ID",
                col
            ]
        ].copy()

        q[
            col
        ] = pd.to_numeric(
            q[col],
            errors="coerce"
        )

        m = G.merge(
            q,
            on="HEX_ID",
            how="left",
            validate="1:1"
        )

        m.plot(
            ax=ax,
            column=col,
            cmap=COEF_CMAP,
            norm=norm,
            edgecolor="none",
            missing_kwds={
                "color": "#eeeeee"
            }
        )

        clean_map_axis(
            ax
        )

        ax.set_title(
            feature.replace(
                "_",
                " "
            ),
            fontweight="bold"
        )

    # bandwidth panel in the final slot
    ax_bw = fig.add_subplot(
        gs[2, 2]
    )

    if "MGWR_BW" in globals():

        bw = MGWR_BW.copy()

        var_col = (
            "VARIABLE"
            if "VARIABLE" in bw.columns
            else bw.columns[0]
        )

        value_col = (
            "BANDWIDTH_N_NEIGHBORS"
            if "BANDWIDTH_N_NEIGHBORS" in bw.columns
            else [
                c
                for c in bw.columns
                if "BAND" in c.upper()
            ][0]
        )

        bw[value_col] = pd.to_numeric(
            bw[value_col],
            errors="coerce"
        )

        bw = bw.dropna(
            subset=[
                value_col
            ]
        )

        bw = bw.sort_values(
            value_col
        )

        ax_bw.barh(
            np.arange(
                len(bw)
            ),
            bw[value_col],
            color=GV[8]
        )

        ax_bw.set_yticks(
            np.arange(
                len(bw)
            ),
            bw[var_col].astype(str)
        )

        ax_bw.set_xscale(
            "log"
        )

        ax_bw.set_xlabel(
            "Adaptive bandwidth\n(nearest neighbors; log scale)"
        )

        ax_bw.set_title(
            "Predictor-specific spatial scales",
            fontweight="bold"
        )

        bw.to_csv(
            SUP_DATA /
            "TABLE_S07_MGWR_BANDWIDTHS.csv",
            index=False
        )

    else:

        ax_bw.text(
            0.5,
            0.5,
            "MGWR bandwidth table\nnot available in memory",
            transform=ax_bw.transAxes,
            ha="center",
            va="center"
        )

        ax_bw.set_axis_off()

    sm = mpl.cm.ScalarMappable(
        norm=norm,
        cmap=COEF_CMAP
    )

    cax = fig.add_axes(
        [
            0.23,
            0.045,
            0.54,
            0.018
        ]
    )

    cb = fig.colorbar(
        sm,
        cax=cax,
        orientation="horizontal"
    )

    cb.set_label(
        "Local MGWR coefficient (°C per +1 SD predictor)"
    )

    fig.suptitle(
        "Figure S7. Full spatial heterogeneity of MGWR local conditional effects",
        fontsize=11.5,
        fontweight="bold"
    )

    supp_save(
        fig,
        "Figure_S07_MGWR_All_Local_Coefficients"
    )

    plt.show()
    plt.close(fig)

    summary_rows = []

    for f, c in beta_map.items():
        v = pd.to_numeric(
            RAW_MGWR[c],
            errors="coerce"
        )

        summary_rows.append({
            "FEATURE": f,
            "MEAN": v.mean(),
            "SD": v.std(),
            "MEDIAN": v.median(),
            "P05": v.quantile(0.05),
            "P95": v.quantile(0.95),
            "PCT_POSITIVE":
                100
                *
                (v > 0).mean(),
        })

    pd.DataFrame(
        summary_rows
    ).to_csv(
        SUP_DATA /
        "TABLE_S07_MGWR_COEFFICIENT_SUMMARY.csv",
        index=False
    )

    register(
        "S7",
        "Full MGWR local-coefficient atlas and predictor bandwidths"
    )


In [ ]:
# ================================================================================================
# FIGURE S8 — GTWR TUNING SURFACE
# ================================================================================================

def _find_gtwr_tuning_supp():
    if "find_gtwr_tuning" in globals():
        try:
            d, rmse_col, source = find_gtwr_tuning()

            if d is not None:
                return d.copy(), rmse_col, source
        except Exception:
            pass

    candidates = []

    for p in Path("/content").rglob("*.csv"):
        if str(p).startswith("/content/drive/"):
            continue

        if "GTWR" not in str(p).upper():
            continue

        try:
            d = pd.read_csv(p)
        except Exception:
            continue

        if not {
            "TAU",
            "K"
        }.issubset(
            d.columns
        ):
            continue

        rmse = None

        for c in [
            "RMSE_C",
            "RMSE"
        ]:
            if c in d.columns:
                rmse = c
                break

        if rmse is None:
            continue

        q = d.copy()

        for c in [
            "TAU",
            "K",
            rmse
        ]:
            q[c] = pd.to_numeric(
                q[c],
                errors="coerce"
            )

        q = q.dropna(
            subset=[
                "TAU",
                "K",
                rmse
            ]
        )

        if (
            q["TAU"].nunique() >= 2
            and
            q["K"].nunique() >= 3
        ):
            candidates.append(
                (
                    q.shape[0],
                    q,
                    rmse,
                    p
                )
            )

    if not candidates:
        return None, None, None

    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )

    _, q, rmse, p = candidates[0]

    return q, rmse, p


GTWR_TUNE_SUPP, GTWR_RMSE_COL, GTWR_TUNE_SOURCE = (
    _find_gtwr_tuning_supp()
)

if GTWR_TUNE_SUPP is None:

    print(
        "S8 skipped: complete GTWR TAU × K tuning table not found."
    )

    register(
        "S8",
        "GTWR parameter tuning",
        status="skipped — tuning table missing"
    )

else:

    D = GTWR_TUNE_SUPP.copy()

    D["TAU"] = pd.to_numeric(
        D["TAU"],
        errors="coerce"
    )

    D["K"] = pd.to_numeric(
        D["K"],
        errors="coerce"
    )

    D[GTWR_RMSE_COL] = pd.to_numeric(
        D[GTWR_RMSE_COL],
        errors="coerce"
    )

    D = D.dropna(
        subset=[
            "TAU",
            "K",
            GTWR_RMSE_COL
        ]
    )

    best = D.loc[
        D[GTWR_RMSE_COL].idxmin()
    ]

    pivot = (
        D.pivot_table(
            index="TAU",
            columns="K",
            values=GTWR_RMSE_COL,
            aggfunc="mean"
        )
        .sort_index()
        .sort_index(axis=1)
    )

    D.to_csv(
        SUP_DATA /
        "TABLE_S08_GTWR_PARAMETER_TUNING.csv",
        index=False
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10.8, 4.3),
        constrained_layout=True
    )

    ax = axes[0]

    im = ax.imshow(
        pivot.to_numpy(float),
        cmap=UNC_CMAP,
        aspect="auto"
    )

    ax.set_xticks(
        range(
            pivot.shape[1]
        ),
        [
            f"{x:g}"
            for x in pivot.columns
        ]
    )

    ax.set_yticks(
        range(
            pivot.shape[0]
        ),
        [
            f"{x:g}"
            for x in pivot.index
        ]
    )

    ax.set_xlabel(
        "Adaptive neighbor count K"
    )

    ax.set_ylabel(
        r"Temporal scale $\tau$"
    )

    ax.set_title(
        "GTWR tuning RMSE surface",
        fontweight="bold"
    )

    for r in range(
        pivot.shape[0]
    ):
        for c in range(
            pivot.shape[1]
        ):
            val = pivot.iloc[
                r,
                c
            ]

            if np.isfinite(val):
                ax.text(
                    c,
                    r,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=6.5
                )

    # mark best
    best_row = list(
        pivot.index
    ).index(
        best["TAU"]
    )

    best_col = list(
        pivot.columns
    ).index(
        best["K"]
    )

    ax.scatter(
        [best_col],
        [best_row],
        marker="*",
        s=95,
        facecolor="white",
        edgecolor="black",
        linewidth=0.9,
        zorder=5
    )

    fig.colorbar(
        im,
        ax=ax,
        shrink=0.80,
        label="RMSE"
    )

    ax2 = axes[1]

    for tau in sorted(
        D["TAU"].unique()
    ):

        q = (
            D.loc[
                D["TAU"].eq(tau)
            ]
            .sort_values("K")
        )

        ax2.plot(
            q["K"],
            q[GTWR_RMSE_COL],
            marker="o",
            ms=3.5,
            lw=1.2,
            label=rf"$\tau$={tau:g}"
        )

    ax2.scatter(
        [best["K"]],
        [best[GTWR_RMSE_COL]],
        marker="*",
        s=90,
        facecolor=GV[7],
        edgecolor="black",
        linewidth=0.8,
        zorder=5
    )

    ax2.set_xlabel(
        "Adaptive neighbor count K"
    )

    ax2.set_ylabel(
        "Tuning RMSE"
    )

    ax2.set_title(
        f"Best: K={best['K']:g}, τ={best['TAU']:g}",
        fontweight="bold"
    )

    ax2.grid(
        axis="y",
        alpha=0.18
    )

    ax2.legend(
        frameon=False,
        ncol=2
    )

    fig.suptitle(
        "Figure S8. Sensitivity of GTWR predictive error to spatial and temporal kernel scales",
        fontsize=11.5,
        fontweight="bold"
    )

    supp_save(
        fig,
        "Figure_S08_GTWR_Tuning_Surface"
    )

    plt.show()
    plt.close(fig)

    register(
        "S8",
        "GTWR spatial-temporal parameter tuning"
    )


In [ ]:
# ================================================================================================
# FIGURE S9 — GGPR UNCERTAINTY CALIBRATION
# ================================================================================================

if not all(
    name in globals()
    for name in [
        "GGPR_SIM",
        "GGPR_REAL"
    ]
):

    print(
        "S9 skipped: GGPR_SIM / GGPR_REAL model-GVSTMR outputs are missing."
    )

    register(
        "S9",
        "GGPR uncertainty calibration",
        status="skipped — GGPR tables missing"
    )

else:

    CAL_ROWS = []
    COVER_ROWS = []

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(10.8, 7.2),
        constrained_layout=True
    )

    for row, (
        case,
        D0
    ) in enumerate(
        [
            ("SIMULATION", GGPR_SIM),
            ("REAL", GGPR_REAL)
        ]
    ):

        D = D0.copy()

        unc_col = detect_uncertainty_col(
            D
        )

        if unc_col is None:
            axes[row, 0].text(
                0.5,
                0.5,
                "Uncertainty column not found",
                transform=axes[row, 0].transAxes,
                ha="center",
                va="center"
            )
            axes[row, 1].text(
                0.5,
                0.5,
                "Uncertainty column not found",
                transform=axes[row, 1].transAxes,
                ha="center",
                va="center"
            )
            continue

        for c in [
            "MODEL_VALUE",
            "OBS_VALUE",
            unc_col,
            "TIME"
        ]:
            D[c] = pd.to_numeric(
                D[c],
                errors="coerce"
            )

        D = D.replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        ).dropna(
            subset=[
                "MODEL_VALUE",
                "OBS_VALUE",
                unc_col,
                "TIME"
            ]
        )

        D["ABS_ERROR"] = np.abs(
            D["MODEL_VALUE"]
            -
            D["OBS_VALUE"]
        )

        # uncertainty deciles
        try:
            D["UNC_BIN"] = pd.qcut(
                D[unc_col],
                10,
                labels=False,
                duplicates="drop"
            )
        except Exception:
            D["UNC_BIN"] = pd.cut(
                D[unc_col],
                10,
                labels=False
            )

        cal = (
            D.groupby(
                "UNC_BIN",
                as_index=False
            )
            .agg(
                MEAN_SD=(
                    unc_col,
                    "mean"
                ),
                MEAN_ABS_ERROR=(
                    "ABS_ERROR",
                    "mean"
                ),
                MEDIAN_ABS_ERROR=(
                    "ABS_ERROR",
                    "median"
                ),
                N=(
                    "ABS_ERROR",
                    "size"
                )
            )
        )

        cal["CASE"] = case

        CAL_ROWS.append(
            cal
        )

        ax = axes[
            row,
            0
        ]

        ax.plot(
            cal["MEAN_SD"],
            cal["MEAN_ABS_ERROR"],
            marker="o",
            lw=1.5,
            color=GV[8]
        )

        lo = min(
            cal["MEAN_SD"].min(),
            cal["MEAN_ABS_ERROR"].min()
        )

        hi = max(
            cal["MEAN_SD"].max(),
            cal["MEAN_ABS_ERROR"].max()
        )

        ax.plot(
            [
                lo,
                hi
            ],
            [
                lo,
                hi
            ],
            ls="--",
            lw=0.8,
            color="#777777"
        )

        ax.set_xlabel(
            "Mean posterior SD"
        )

        ax.set_ylabel(
            "Mean absolute error"
        )

        ax.set_title(
            (
                "Simulation"
                if case == "SIMULATION"
                else "Real case"
            )
            +
            ": uncertainty–error calibration",
            fontweight="bold"
        )

        ax.grid(
            alpha=0.18
        )

        # empirical 95% coverage by time
        D["CI95_LO"] = (
            D["MODEL_VALUE"]
            -
            1.96
            *
            D[unc_col]
        )

        D["CI95_HI"] = (
            D["MODEL_VALUE"]
            +
            1.96
            *
            D[unc_col]
        )

        D["COVERED"] = (
            D["OBS_VALUE"].ge(
                D["CI95_LO"]
            )
            &
            D["OBS_VALUE"].le(
                D["CI95_HI"]
            )
        ).astype(float)

        cov = (
            D.groupby(
                "TIME",
                as_index=False
            )
            .agg(
                COVERAGE=(
                    "COVERED",
                    "mean"
                ),
                MEAN_SD=(
                    unc_col,
                    "mean"
                ),
                N=(
                    "COVERED",
                    "size"
                )
            )
        )

        cov["CASE"] = case

        COVER_ROWS.append(
            cov
        )

        ax2 = axes[
            row,
            1
        ]

        ax2.plot(
            cov["TIME"],
            cov["COVERAGE"],
            marker="o",
            lw=1.5,
            color=GV[6]
        )

        ax2.axhline(
            0.95,
            color="#777777",
            lw=0.8,
            ls="--"
        )

        ax2.set_ylim(
            0,
            1.02
        )

        ax2.set_xlabel(
            "Time"
            if case == "SIMULATION"
            else "Year"
        )

        ax2.set_ylabel(
            "Empirical 95% interval coverage"
        )

        ax2.set_title(
            (
                "Simulation"
                if case == "SIMULATION"
                else "Real case"
            )
            +
            ": predictive-interval coverage",
            fontweight="bold"
        )

        ax2.grid(
            axis="y",
            alpha=0.18
        )

    if CAL_ROWS:
        pd.concat(
            CAL_ROWS,
            ignore_index=True
        ).to_csv(
            SUP_DATA /
            "TABLE_S09_GGPR_UNCERTAINTY_CALIBRATION.csv",
            index=False
        )

    if COVER_ROWS:
        pd.concat(
            COVER_ROWS,
            ignore_index=True
        ).to_csv(
            SUP_DATA /
            "TABLE_S09_GGPR_CI95_COVERAGE_BY_TIME.csv",
            index=False
        )

    fig.suptitle(
        "Figure S9. Calibration of GGPR predictive uncertainty against realized model error",
        fontsize=11.5,
        fontweight="bold"
    )

    supp_save(
        fig,
        "Figure_S09_GGPR_Uncertainty_Calibration"
    )

    plt.show()
    plt.close(fig)

    register(
        "S9",
        "GGPR uncertainty–error calibration and empirical 95% coverage"
    )


In [ ]:
# ================================================================================================
# FIGURE S10 — ORIGINAL GEOSHAPLEY / SHAP-STYLE SUMMARY PLOTS
#
# Replaces the previous heatmap.
#
# This follows the plotting grammar of GeoShapley's own summary_plot():
#   x-position = GeoShapley contribution
#   y-order    = mean absolute contribution
#   point pile = contribution density
#   color      = original feature value (Low -> High)
#
# Simulation: period-integrated final GeoShapley table from the simulation ZIP.
# Real case : all annual 2015–2024 GeoShapley values pooled across hexagon-years.
# ================================================================================================

import sys
import subprocess
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap, Normalize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

try:
    import shap
except Exception:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "shap"]
    )
    import shap


# Original SHAP/GeoShapley-style blue -> purple -> magenta scale
SHAP_CMAP = LinearSegmentedColormap.from_list(
    "GeoShapley_SHAP",
    [
        "#008BFB",
        "#6D55B4",
        "#FF0051",
    ]
)


REAL_FEATURES_S10 = [
    "TEMP_C",
    "NDVI",
    "SOIL_MM",
    "RH",
    "PRECIP_MM",
    "ELEV_M",
    "SW_RAD_WM2",
    "WIND_MS",
]


def _norm_token(x):
    return re.sub(
        r"[^A-Z0-9]",
        "",
        str(x).upper()
    )


def _find_geoshap_component(df, feature):
    """
    Match forms such as:
      GEOSHAP_TEMP_C
      GEOSHAP_TEMP_C_C
      GEOSHAP_GEO
      GEOSHAP_GEO_C
    """
    target = _norm_token(feature)

    candidates = [
        c
        for c in df.columns
        if str(c).upper().startswith("GEOSHAP_")
    ]

    for c in candidates:
        base = re.sub(
            r"^GEOSHAP_",
            "",
            str(c),
            flags=re.I
        )

        nb = _norm_token(base)

        if nb == target:
            return c

        if nb == _norm_token(feature + "_C"):
            return c

    return None


def _make_explanation(
    df,
    feature_names,
    component_names,
    add_geo=False,
    geo_component=None,
    max_points=6000,
    random_state=123
):
    needed = list(feature_names) + list(component_names)

    if add_geo and geo_component is not None:
        needed.append(geo_component)

    q = df[
        [
            c
            for c in needed
            if c in df.columns
        ]
    ].copy()

    # numeric coercion
    for c in q.columns:
        q[c] = pd.to_numeric(
            q[c],
            errors="coerce"
        )

    ok = q[
        component_names
    ].notna().all(axis=1)

    if add_geo and geo_component is not None:
        ok &= q[geo_component].notna()

    q = q.loc[ok].copy()

    if len(q) > max_points:
        q = q.sample(
            max_points,
            random_state=random_state
        )

    values = q[
        component_names
    ].to_numpy(float)

    data = q[
        feature_names
    ].to_numpy(float)

    names = list(feature_names)

    # Fill occasional missing raw feature values only for coloring.
    for j in range(data.shape[1]):
        med = np.nanmedian(
            data[:, j]
        )

        if not np.isfinite(med):
            med = 0.0

        bad = ~np.isfinite(
            data[:, j]
        )

        data[
            bad,
            j
        ] = med

    # GeoShapley's own summary_plot uses GEO feature value = 0.
    if (
        add_geo
        and
        geo_component is not None
    ):
        values = np.column_stack(
            [
                values,
                pd.to_numeric(
                    q[geo_component],
                    errors="coerce"
                ).to_numpy(float)
            ]
        )

        data = np.column_stack(
            [
                data,
                np.zeros(
                    len(q),
                    dtype=float
                )
            ]
        )

        names.append(
            "GEO"
        )

    return shap.Explanation(
        values=values,
        data=data,
        feature_names=names
    )


def _draw_original_geoshap_beeswarm(
    ax,
    explanation,
    title,
    panel_letter
):
    """
    Draw a GeoShapley summary plot in the same visual grammar
    as the original GeoShapley/SHAP beeswarm.
    """

    # Preferred recent SHAP API
    try:
        shap.plots.beeswarm(
            explanation,
            max_display=len(
                explanation.feature_names
            ),
            order=shap.Explanation.abs.mean(0),
            color=SHAP_CMAP,
            alpha=0.78,
            s=10,
            ax=ax,
            show=False,
            color_bar=False,
            plot_size=None,
            group_remaining_features=False
        )

    # Older SHAP fallback
    except TypeError:
        plt.sca(ax)

        shap.summary_plot(
            explanation.values,
            explanation.data,
            feature_names=explanation.feature_names,
            plot_type="dot",
            color=SHAP_CMAP,
            show=False,
            color_bar=False,
            plot_size=None,
            max_display=len(
                explanation.feature_names
            )
        )

    ax.axvline(
        0,
        color="#777777",
        lw=0.75,
        zorder=0
    )

    ax.set_title(
        title,
        fontsize=10.0,
        fontweight="bold",
        pad=8
    )

    ax.set_xlabel(
        "GeoShapley value (impact on GGPR output, °C)",
        fontsize=8.0
    )

    ax.tick_params(
        axis="x",
        labelsize=7.2
    )

    ax.tick_params(
        axis="y",
        labelsize=7.5,
        length=0
    )

    for sp in [
        "top",
        "right",
        "left"
    ]:
        ax.spines[
            sp
        ].set_visible(
            False
        )

    ax.grid(
        axis="y",
        alpha=0.10,
        lw=0.5
    )

    ax.text(
        -0.08,
        1.03,
        panel_letter,
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        ha="left",
        va="bottom"
    )

    # True Low -> High feature-value bar, like the original GeoShapley plot.
    cax = inset_axes(
        ax,
        width="2.4%",
        height="82%",
        loc="center right",
        bbox_to_anchor=(
            0.10,
            0.0,
            1.0,
            1.0
        ),
        bbox_transform=ax.transAxes,
        borderpad=0
    )

    sm = mpl.cm.ScalarMappable(
        cmap=SHAP_CMAP,
        norm=Normalize(
            0,
            1
        )
    )

    cb = plt.colorbar(
        sm,
        cax=cax
    )

    cb.set_ticks(
        [
            0,
            1
        ]
    )

    cb.set_ticklabels(
        [
            "Low",
            "High"
        ]
    )

    cb.set_label(
        "Feature value",
        fontsize=7.0,
        labelpad=3
    )

    cb.ax.tick_params(
        labelsize=6.6,
        length=0
    )

    cb.outline.set_visible(
        False
    )


# ------------------------------------------------------------------------------------------------
# A. SIMULATION — use the final period-integrated analytical GeoShapley table already in SIM ZIP
# ------------------------------------------------------------------------------------------------

SIM_S10_PATH = extract_by_suffix(
    SIM_ZIP,
    "GEOSHAPLEY_ALL_HEX.parquet",
    "sim_s10"
)

SIM_S10 = pd.read_parquet(
    SIM_S10_PATH
)

SIM_S10[
    "HEX_ID"
] = SIM_S10[
    "HEX_ID"
].astype(str)

# If X itself is not in the GeoShapley table, merge it from the simulation panel.
if "X" not in SIM_S10.columns:

    SIM_X = (
        SIM_PANEL[
            [
                "HEX_ID",
                "X"
            ]
        ]
        .groupby(
            "HEX_ID",
            as_index=False
        )
        .agg(
            X=(
                "X",
                "mean"
            )
        )
    )

    SIM_X[
        "HEX_ID"
    ] = SIM_X[
        "HEX_ID"
    ].astype(str)

    SIM_S10 = SIM_S10.merge(
        SIM_X,
        on="HEX_ID",
        how="left",
        validate="1:1"
    )


sim_x_geo = _find_geoshap_component(
    SIM_S10,
    "X"
)

sim_geo_geo = _find_geoshap_component(
    SIM_S10,
    "GEO"
)

if sim_x_geo is None:
    raise RuntimeError(
        "Figure S10: GEOSHAP_X component was not found in the simulation GeoShapley table."
    )

SIM_EXPL = _make_explanation(
    SIM_S10,
    feature_names=[
        "X"
    ],
    component_names=[
        sim_x_geo
    ],
    add_geo=(
        sim_geo_geo is not None
    ),
    geo_component=sim_geo_geo,
    max_points=6000
)


# ------------------------------------------------------------------------------------------------
# B. REAL CASE — pool all annual 2015–2024 hexagon-years
# ------------------------------------------------------------------------------------------------

REAL_S10 = pd.concat(
    [
        ANNUAL_MAPS[
            year
        ].drop(
            columns=[
                "geometry"
            ],
            errors="ignore"
        )
        for year in sorted(
            ANNUAL_MAPS
        )
    ],
    ignore_index=True
)

real_features = []
real_components = []

for f in REAL_FEATURES_S10:

    gc = _find_geoshap_component(
        REAL_S10,
        f
    )

    if (
        gc is not None
        and
        f in REAL_S10.columns
    ):
        real_features.append(
            f
        )

        real_components.append(
            gc
        )


real_geo_component = _find_geoshap_component(
    REAL_S10,
    "GEO"
)

if len(real_features) < 6:
    raise RuntimeError(
        "Figure S10: fewer than six real-case environmental GeoShapley components were matched."
    )


REAL_EXPL = _make_explanation(
    REAL_S10,
    feature_names=real_features,
    component_names=real_components,
    add_geo=(
        real_geo_component is not None
    ),
    geo_component=real_geo_component,
    max_points=7000
)


# ------------------------------------------------------------------------------------------------
# C. FINAL FIGURE
# ------------------------------------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        13.2,
        5.9
    ),
    gridspec_kw={
        "width_ratios": [
            0.75,
            1.25
        ]
    }
)

fig.subplots_adjust(
    left=0.08,
    right=0.94,
    top=0.86,
    bottom=0.14,
    wspace=0.42
)

_draw_original_geoshap_beeswarm(
    axes[
        0
    ],
    SIM_EXPL,
    "Simulation case",
    "a"
)

_draw_original_geoshap_beeswarm(
    axes[
        1
    ],
    REAL_EXPL,
    "Real case: 2015–2024",
    "b"
)

fig.suptitle(
    "Figure S10. GeoShapley summary distributions of feature impacts on GGPR predictions",
    fontsize=12.0,
    fontweight="bold",
    y=0.965
)

fig.text(
    0.5,
    0.035,
    "Points are individual spatial observations; horizontal position is the GeoShapley contribution and color denotes the corresponding feature value.",
    ha="center",
    va="center",
    fontsize=7.4,
    color="#555555"
)

supp_save(
    fig,
    "Figure_S10_GeoShapley_Original_Style_Summary"
)

plt.show()
plt.close(
    fig
)

register(
    "S10",
    "Original-style GeoShapley summary distributions for simulation and real case"
)


In [ ]:
# ================================================================================================
# FIGURE S11 — AIR-TEMPERATURE ABLATION + INDEPENDENT NOAA USCRN VALIDATION
#
# FIXED fresh-runtime version.
#
# The previous cell failed because it looked for already-saved ablation CSVs.
# Those CSVs are not one of the three required ZIP inputs.
#
# The REAL ZIP DOES contain the exact required independent-validation inputs:
#     USCRN_2015_2024.parquet
#     STX_2015_2024.parquet
#
# This cell therefore loads those two tables and reruns the already-established
# M8-vs-M7 ablation workflow. It does not download rasters and does not refit
# MGWR, GTWR or GGPR.
# ================================================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
from IPython.display import display

# ------------------------------------------------------------------------------------------------
# 1. LOAD EXACT SAVED USCRN / STATION-PREDICTOR TABLES FROM THE REQUIRED REAL ZIP
# ------------------------------------------------------------------------------------------------

USCRN_PATH_S11 = extract_by_suffix(
    REAL_ZIP,
    "USCRN_2015_2024.parquet",
    "real_s11"
)

STX_PATH_S11 = extract_by_suffix(
    REAL_ZIP,
    "STX_2015_2024.parquet",
    "real_s11"
)

USCRN = pd.read_parquet(
    USCRN_PATH_S11
)

STX = pd.read_parquet(
    STX_PATH_S11
)

for d in [
    USCRN,
    STX
]:
    if "WBANNO" in d.columns:
        d[
            "WBANNO"
        ] = (
            d[
                "WBANNO"
            ]
            .astype(str)
            .str.replace(
                r"\.0$",
                "",
                regex=True
            )
            .str.zfill(
                5
            )
        )

    if "YEAR" in d.columns:
        d[
            "YEAR"
        ] = pd.to_numeric(
            d[
                "YEAR"
            ],
            errors="coerce"
        ).astype(
            "Int64"
        )


# ------------------------------------------------------------------------------------------------
# 2. ENSURE PANEL HAS THE EXACT 30-km USCRN TRAINING EXCLUSION
# ------------------------------------------------------------------------------------------------

PANEL[
    "HEX_ID"
] = PANEL[
    "HEX_ID"
].astype(str)

HEX[
    "HEX_ID"
] = HEX[
    "HEX_ID"
].astype(str)


if (
    "GROUND_EXCLUSION" not in PANEL.columns
):

    # Prefer the stored exclusion flag if it already exists in HEX.
    if (
        "GROUND_EXCLUSION" in HEX.columns
    ):

        PANEL = PANEL.merge(
            HEX[
                [
                    "HEX_ID",
                    "GROUND_EXCLUSION"
                ]
            ].drop_duplicates(
                "HEX_ID"
            ),
            on="HEX_ID",
            how="left",
            validate="many_to_one"
        )

    else:

        # Reconstruct the originally used 30-km station exclusion exactly.
        lon_col = (
            "LON"
            if "LON" in USCRN.columns
            else "LONGITUDE"
        )

        lat_col = (
            "LAT"
            if "LAT" in USCRN.columns
            else "LATITUDE"
        )

        station_locations = (
            USCRN[
                [
                    "WBANNO",
                    lon_col,
                    lat_col
                ]
            ]
            .drop_duplicates(
                "WBANNO"
            )
            .rename(
                columns={
                    lon_col: "LON",
                    lat_col: "LAT"
                }
            )
            .dropna(
                subset=[
                    "LON",
                    "LAT"
                ]
            )
        )

        station_gdf = gpd.GeoDataFrame(
            station_locations,
            geometry=gpd.points_from_xy(
                station_locations[
                    "LON"
                ],
                station_locations[
                    "LAT"
                ]
            ),
            crs="EPSG:4326"
        ).to_crs(
            "EPSG:5070"
        )

        # Ensure projected hex coordinates are available.
        if (
            "X_KM" not in HEX.columns
            or
            "Y_KM" not in HEX.columns
        ):

            H = HEX.to_crs(
                "EPSG:5070"
            ).copy()

            cent = H.geometry.centroid

            HEX[
                "X_KM"
            ] = cent.x.to_numpy() / 1000.0

            HEX[
                "Y_KM"
            ] = cent.y.to_numpy() / 1000.0

        station_xy_km = np.column_stack(
            [
                station_gdf.geometry.x.to_numpy(float) / 1000.0,
                station_gdf.geometry.y.to_numpy(float) / 1000.0,
            ]
        )

        hex_xy_km = HEX[
            [
                "X_KM",
                "Y_KM"
            ]
        ].to_numpy(float)

        tree = cKDTree(
            station_xy_km
        )

        dist_km, _ = tree.query(
            hex_xy_km,
            k=1
        )

        HEX[
            "DIST_TO_USCRN_KM"
        ] = dist_km

        HEX[
            "GROUND_EXCLUSION"
        ] = (
            HEX[
                "DIST_TO_USCRN_KM"
            ]
            <=
            30.0
        )

        PANEL = PANEL.merge(
            HEX[
                [
                    "HEX_ID",
                    "GROUND_EXCLUSION"
                ]
            ],
            on="HEX_ID",
            how="left",
            validate="many_to_one"
        )


PANEL[
    "GROUND_EXCLUSION"
] = (
    PANEL[
        "GROUND_EXCLUSION"
    ]
    .fillna(
        False
    )
    .astype(
        bool
    )
)


# Ensure the predictor panel has X_KM / Y_KM, needed for common geographic CV blocks.
if (
    "X_KM" not in PANEL.columns
    or
    "Y_KM" not in PANEL.columns
):

    if (
        "X_KM" not in HEX.columns
        or
        "Y_KM" not in HEX.columns
    ):

        H = HEX.to_crs(
            "EPSG:5070"
        ).copy()

        cent = H.geometry.centroid

        HEX[
            "X_KM"
        ] = cent.x.to_numpy() / 1000.0

        HEX[
            "Y_KM"
        ] = cent.y.to_numpy() / 1000.0

    PANEL = PANEL.merge(
        HEX[
            [
                "HEX_ID",
                "X_KM",
                "Y_KM"
            ]
        ].drop_duplicates(
            "HEX_ID"
        ),
        on="HEX_ID",
        how="left",
        validate="many_to_one"
    )


# Save all exact ablation products inside the Supplementary package.
OUT = SUP_DATA / "AIR_TEMPERATURE_ABLATION"

OUT.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "USCRN table:",
    USCRN.shape
)

print(
    "STX table:",
    STX.shape
)

print(
    "Training-exclusion hexes:",
    int(
        PANEL[
            [
                "HEX_ID",
                "GROUND_EXCLUSION"
            ]
        ]
        .drop_duplicates(
            "HEX_ID"
        )[
            "GROUND_EXCLUSION"
        ]
        .sum()
    )
)


# ======================================================================================
# GVSTMR — AIR-TEMPERATURE ABLATION TEST
# M8 = full 8-predictor model
# M7 = same model WITHOUT TEMP_C
# Period: 2015–2024
#
# Uses existing runtime objects:
#   PANEL, STX, USCRN, OUT
#
# No raster download / no annual extraction / no MGWR-GTWR-GGPR
# ======================================================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)

# ----------------------------------------------------------------------
# SETTINGS
# ----------------------------------------------------------------------

YEARS = list(range(2015, 2025))

YCOL = "LST_MEAN_C"

FEATURES_M8 = [
    "TEMP_C",
    "SW_RAD_WM2",
    "RH",
    "WIND_MS",
    "PRECIP_MM",
    "NDVI",
    "SOIL_MM",
    "ELEV_M",
]

FEATURES_M7 = [
    "SW_RAD_WM2",
    "RH",
    "WIND_MS",
    "PRECIP_MM",
    "NDVI",
    "SOIL_MM",
    "ELEV_M",
]

RANDOM_STATE = 123
N_BLOCKS = 5


print("=" * 95)
print("GVSTMR — AIR TEMPERATURE ABLATION")
print("M8 = WITH TEMP_C")
print("M7 = WITHOUT TEMP_C")
print("=" * 95)


# ======================================================================================
# 1. CHECK OBJECTS
# ======================================================================================

needed = [
    "PANEL",
    "STX",
    "USCRN",
    "OUT",
]

missing = [
    x for x in needed
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing runtime objects: {missing}\n"
        "Run the panel/USCRN recovery cell first."
    )


# ======================================================================================
# 2. COMMON TRAINING SAMPLE
#
# Important:
# Both models use EXACTLY the same rows.
# USCRN-nearby hexagons remain excluded.
# ======================================================================================

TRAIN = PANEL.loc[
    (~PANEL["GROUND_EXCLUSION"].astype(bool))
    &
    PANEL[YCOL].notna()
].copy()


print(
    "\nTraining rows:",
    f"{len(TRAIN):,}"
)

print(
    "Training hexes:",
    TRAIN["HEX_ID"].nunique()
)


# ======================================================================================
# 3. COMMON GEOGRAPHIC BLOCKS
#
# Create block assignment ONCE.
# Both M8 and M7 use exactly the same test regions.
# ======================================================================================

HEX_LOC = (
    TRAIN[
        [
            "HEX_ID",
            "X_KM",
            "Y_KM",
        ]
    ]
    .drop_duplicates("HEX_ID")
    .copy()
)


km = KMeans(
    n_clusters=N_BLOCKS,
    random_state=RANDOM_STATE,
    n_init=30,
)


HEX_LOC["BLOCK"] = km.fit_predict(
    HEX_LOC[
        [
            "X_KM",
            "Y_KM",
        ]
    ]
)


CV_DATA = TRAIN.merge(
    HEX_LOC[
        [
            "HEX_ID",
            "BLOCK",
        ]
    ],
    on="HEX_ID",
    how="left",
    validate="many_to_one",
)


print("\nCommon geographic blocks:")

display(
    CV_DATA
    .groupby("BLOCK")
    .agg(
        N_ROWS=("HEX_ID", "size"),
        N_HEX=("HEX_ID", "nunique"),
    )
)


# ======================================================================================
# 4. COMMON MODEL FACTORY
# ======================================================================================

def make_ols():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LinearRegression()
        ),
    ])


# ======================================================================================
# 5. METRIC FUNCTION
# ======================================================================================

def metrics(
    obs,
    pred,
):

    obs = np.asarray(
        obs,
        dtype=float
    )

    pred = np.asarray(
        pred,
        dtype=float
    )

    ok = (
        np.isfinite(obs)
        &
        np.isfinite(pred)
    )

    obs = obs[ok]
    pred = pred[ok]

    return {
        "N": len(obs),

        "R2": float(
            r2_score(
                obs,
                pred
            )
        ),

        "RMSE_C": float(
            mean_squared_error(
                obs,
                pred
            ) ** 0.5
        ),

        "MAE_C": float(
            mean_absolute_error(
                obs,
                pred
            )
        ),

        "BIAS_C": float(
            np.mean(
                pred - obs
            )
        ),

        "PEARSON_R": float(
            np.corrcoef(
                obs,
                pred
            )[0, 1]
        ),
    }


# ======================================================================================
# 6. FIT BOTH MODELS
# ======================================================================================

models = {
    "M8_WITH_TEMP": FEATURES_M8,
    "M7_NO_TEMP": FEATURES_M7,
}


FIT_ROWS = []
CV_ROWS = []
COEF_ROWS = []

FITTED_MODELS = {}


for model_name, feats in models.items():

    print("\n" + "=" * 95)
    print(model_name)
    print("Predictors:", feats)
    print("=" * 95)

    model = make_ols()

    model.fit(
        TRAIN[feats],
        TRAIN[YCOL]
    )

    FITTED_MODELS[
        model_name
    ] = model


    # ------------------------------------------------------------------
    # In-sample fit
    # ------------------------------------------------------------------

    pred_fit = model.predict(
        TRAIN[feats]
    )

    m = metrics(
        TRAIN[YCOL],
        pred_fit
    )

    n = len(TRAIN)
    p = len(feats)

    adj_r2 = (
        1
        -
        (1 - m["R2"])
        *
        (n - 1)
        /
        (n - p - 1)
    )


    FIT_ROWS.append({
        "MODEL": model_name,
        "N_PREDICTORS": p,
        "N_ROWS": n,
        "FIT_R2": m["R2"],
        "FIT_ADJ_R2": adj_r2,
        "FIT_RMSE_C": m["RMSE_C"],
        "FIT_MAE_C": m["MAE_C"],
    })


    # ------------------------------------------------------------------
    # X-standardized coefficients
    # ------------------------------------------------------------------

    coefs = (
        model
        .named_steps["model"]
        .coef_
    )


    for v, beta in zip(
        feats,
        coefs
    ):

        COEF_ROWS.append({
            "MODEL": model_name,
            "VARIABLE": v,
            "BETA_C_PER_1SD_X": float(beta),
            "ABS_BETA": abs(float(beta)),
        })


    # ------------------------------------------------------------------
    # SAME spatial blocks
    # ------------------------------------------------------------------

    for b in range(
        N_BLOCKS
    ):

        tr = (
            CV_DATA["BLOCK"]
            != b
        )

        te = (
            CV_DATA["BLOCK"]
            == b
        )


        cv_model = make_ols()


        cv_model.fit(
            CV_DATA.loc[
                tr,
                feats
            ],
            CV_DATA.loc[
                tr,
                YCOL
            ]
        )


        cv_pred = cv_model.predict(
            CV_DATA.loc[
                te,
                feats
            ]
        )


        cv_obs = CV_DATA.loc[
            te,
            YCOL
        ]


        cm = metrics(
            cv_obs,
            cv_pred
        )


        CV_ROWS.append({
            "MODEL": model_name,
            "BLOCK": b,
            "N_ROWS": int(
                te.sum()
            ),
            "N_HEX": int(
                CV_DATA.loc[
                    te,
                    "HEX_ID"
                ].nunique()
            ),
            "R2": cm["R2"],
            "RMSE_C": cm["RMSE_C"],
            "MAE_C": cm["MAE_C"],
            "BIAS_C": cm["BIAS_C"],
        })


# ======================================================================================
# 7. FIT TABLE
# ======================================================================================

FIT_COMPARE = pd.DataFrame(
    FIT_ROWS
)


print("\n" + "=" * 95)
print("NATIONAL FIT — M8 vs M7")
print("=" * 95)

display(
    FIT_COMPARE
)


# ======================================================================================
# 8. SPATIAL CV TABLE
# ======================================================================================

CV_COMPARE = pd.DataFrame(
    CV_ROWS
)


print("\n" + "=" * 95)
print("SPATIAL BLOCK CV — ALL BLOCKS")
print("=" * 95)

display(
    CV_COMPARE
)


CV_SUMMARY = (
    CV_COMPARE
    .groupby(
        "MODEL",
        as_index=False
    )
    .agg(
        SPATIAL_CV_MEAN_R2=(
            "R2",
            "mean"
        ),
        SPATIAL_CV_MEDIAN_R2=(
            "R2",
            "median"
        ),
        SPATIAL_CV_MIN_R2=(
            "R2",
            "min"
        ),
        SPATIAL_CV_MAX_R2=(
            "R2",
            "max"
        ),
        SPATIAL_CV_MEAN_RMSE_C=(
            "RMSE_C",
            "mean"
        ),
        SPATIAL_CV_MEAN_MAE_C=(
            "MAE_C",
            "mean"
        ),
    )
)


print("\n" + "=" * 95)
print("SPATIAL CV SUMMARY")
print("=" * 95)

display(
    CV_SUMMARY
)


# ======================================================================================
# 9. PREPARE USCRN VALIDATION
# ======================================================================================

U = USCRN.copy()
S = STX.copy()


for df in [
    U,
    S
]:

    df["WBANNO"] = (
        df["WBANNO"]
        .astype(str)
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
        .str.zfill(5)
    )

    df["YEAR"] = (
        pd.to_numeric(
            df["YEAR"],
            errors="coerce"
        )
        .astype("Int64")
    )


VAL = U.merge(
    S,
    on=[
        "WBANNO",
        "YEAR"
    ],
    how="inner",
    validate="one_to_one"
)


print(
    "\nUSCRN station-years:",
    len(VAL)
)

print(
    "USCRN stations:",
    VAL["WBANNO"].nunique()
)


# ======================================================================================
# 10. USCRN VALIDATION — M8 vs M7
# ======================================================================================

GROUND_ROWS = []


for model_name, feats in models.items():

    model = FITTED_MODELS[
        model_name
    ]


    pred = model.predict(
        VAL[feats]
    )


    VAL[
        f"PRED_{model_name}"
    ] = pred


    gm = metrics(
        VAL[
            "GROUND_SUR_TEMP_C"
        ],
        pred
    )


    GROUND_ROWS.append({
        "MODEL": model_name,
        **gm
    })


# Raw MODIS reference
raw = metrics(
    VAL[
        "GROUND_SUR_TEMP_C"
    ],
    VAL[
        YCOL
    ]
)


GROUND_ROWS.append({
    "MODEL": "RAW_MODIS",
    **raw
})


GROUND_COMPARE = pd.DataFrame(
    GROUND_ROWS
)


print("\n" + "=" * 95)
print("INDEPENDENT NOAA USCRN VALIDATION")
print("=" * 95)

display(
    GROUND_COMPARE
)


# ======================================================================================
# 11. GROUND VALIDATION BY YEAR
# ======================================================================================

GROUND_YEAR_ROWS = []


for year in YEARS:

    d = VAL.loc[
        VAL[
            "YEAR"
        ].eq(year)
    ].copy()


    if len(d) < 10:
        continue


    for model_name, feats in models.items():

        pred_col = (
            f"PRED_{model_name}"
        )


        yy = metrics(
            d[
                "GROUND_SUR_TEMP_C"
            ],
            d[
                pred_col
            ]
        )


        GROUND_YEAR_ROWS.append({
            "YEAR": year,
            "MODEL": model_name,
            **yy
        })


    rr = metrics(
        d[
            "GROUND_SUR_TEMP_C"
        ],
        d[
            YCOL
        ]
    )


    GROUND_YEAR_ROWS.append({
        "YEAR": year,
        "MODEL": "RAW_MODIS",
        **rr
    })


GROUND_BY_YEAR = pd.DataFrame(
    GROUND_YEAR_ROWS
)


print("\n" + "=" * 95)
print("USCRN VALIDATION BY YEAR")
print("=" * 95)

display(
    GROUND_BY_YEAR
)


# ======================================================================================
# 12. COEFFICIENTS
# ======================================================================================

COEF_COMPARE = (
    pd.DataFrame(
        COEF_ROWS
    )
    .sort_values(
        [
            "MODEL",
            "ABS_BETA"
        ],
        ascending=[
            True,
            False
        ]
    )
)


print("\n" + "=" * 95)
print("COEFFICIENTS")
print("=" * 95)

display(
    COEF_COMPARE
)


# ======================================================================================
# 13. DIRECT ABLATION SUMMARY
# ======================================================================================

M8_FIT = FIT_COMPARE.loc[
    FIT_COMPARE[
        "MODEL"
    ].eq(
        "M8_WITH_TEMP"
    )
].iloc[0]


M7_FIT = FIT_COMPARE.loc[
    FIT_COMPARE[
        "MODEL"
    ].eq(
        "M7_NO_TEMP"
    )
].iloc[0]


M8_CV = CV_SUMMARY.loc[
    CV_SUMMARY[
        "MODEL"
    ].eq(
        "M8_WITH_TEMP"
    )
].iloc[0]


M7_CV = CV_SUMMARY.loc[
    CV_SUMMARY[
        "MODEL"
    ].eq(
        "M7_NO_TEMP"
    )
].iloc[0]


M8_G = GROUND_COMPARE.loc[
    GROUND_COMPARE[
        "MODEL"
    ].eq(
        "M8_WITH_TEMP"
    )
].iloc[0]


M7_G = GROUND_COMPARE.loc[
    GROUND_COMPARE[
        "MODEL"
    ].eq(
        "M7_NO_TEMP"
    )
].iloc[0]


ABLATION = pd.DataFrame([{

    "PERIOD": "2015-2024",

    "M8_FIT_R2":
        M8_FIT["FIT_R2"],

    "M7_FIT_R2":
        M7_FIT["FIT_R2"],

    "DELTA_FIT_R2_M8_MINUS_M7":
        M8_FIT["FIT_R2"]
        -
        M7_FIT["FIT_R2"],


    "M8_SPATIAL_CV_MEAN_R2":
        M8_CV[
            "SPATIAL_CV_MEAN_R2"
        ],

    "M7_SPATIAL_CV_MEAN_R2":
        M7_CV[
            "SPATIAL_CV_MEAN_R2"
        ],

    "DELTA_SPATIAL_CV_R2_M8_MINUS_M7":
        M8_CV[
            "SPATIAL_CV_MEAN_R2"
        ]
        -
        M7_CV[
            "SPATIAL_CV_MEAN_R2"
        ],


    "M8_USCRN_R2":
        M8_G["R2"],

    "M7_USCRN_R2":
        M7_G["R2"],

    "DELTA_USCRN_R2_M8_MINUS_M7":
        M8_G["R2"]
        -
        M7_G["R2"],


    "M8_USCRN_RMSE_C":
        M8_G["RMSE_C"],

    "M7_USCRN_RMSE_C":
        M7_G["RMSE_C"],

    "DELTA_USCRN_RMSE_C_M7_MINUS_M8":
        M7_G["RMSE_C"]
        -
        M8_G["RMSE_C"],
}])


print("\n" + "=" * 95)
print("AIR-TEMPERATURE ABLATION — FINAL SUMMARY")
print("=" * 95)

display(
    ABLATION
)


# ======================================================================================
# 14. SAVE
# ======================================================================================

FIT_COMPARE.to_csv(
    OUT /
    "AIR_TEMP_ABLATION_FIT_2015_2024.csv",
    index=False
)


CV_COMPARE.to_csv(
    OUT /
    "AIR_TEMP_ABLATION_SPATIAL_CV_BLOCKS_2015_2024.csv",
    index=False
)


CV_SUMMARY.to_csv(
    OUT /
    "AIR_TEMP_ABLATION_SPATIAL_CV_SUMMARY_2015_2024.csv",
    index=False
)


GROUND_COMPARE.to_csv(
    OUT /
    "AIR_TEMP_ABLATION_USCRN_2015_2024.csv",
    index=False
)


GROUND_BY_YEAR.to_csv(
    OUT /
    "AIR_TEMP_ABLATION_USCRN_BY_YEAR_2015_2024.csv",
    index=False
)


COEF_COMPARE.to_csv(
    OUT /
    "AIR_TEMP_ABLATION_COEFFICIENTS_2015_2024.csv",
    index=False
)


ABLATION.to_csv(
    OUT /
    "AIR_TEMP_ABLATION_FINAL_SUMMARY_2015_2024.csv",
    index=False
)


VAL.to_parquet(
    OUT /
    "AIR_TEMP_ABLATION_USCRN_PREDICTIONS_2015_2024.parquet",
    index=False
)


print("\n✅ ABLATION COMPLETE")
print("Saved to:")
print(OUT)


# ================================================================================================
# S11 PUBLICATION FIGURE
# ================================================================================================

import matplotlib.pyplot as plt
import numpy as np

M8_COLOR = "#C23B9B"
M7_COLOR = "#55B4C8"
RAW_COLOR = "#555555"

# Exact final summary already produced above.
r = ABLATION.iloc[
    0
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        13.6,
        4.7
    )
)

fig.subplots_adjust(
    left=0.065,
    right=0.985,
    top=0.82,
    bottom=0.20,
    wspace=0.34
)


# ------------------------------------------------------------------------------------------------
# a. NATIONAL / SPATIAL-CV / INDEPENDENT USCRN R2
# ------------------------------------------------------------------------------------------------

ax = axes[
    0
]

labels = [
    "National fit",
    "Spatial CV",
    "USCRN"
]

m8 = np.array(
    [
        r[
            "M8_FIT_R2"
        ],
        r[
            "M8_SPATIAL_CV_MEAN_R2"
        ],
        r[
            "M8_USCRN_R2"
        ],
    ],
    dtype=float
)

m7 = np.array(
    [
        r[
            "M7_FIT_R2"
        ],
        r[
            "M7_SPATIAL_CV_MEAN_R2"
        ],
        r[
            "M7_USCRN_R2"
        ],
    ],
    dtype=float
)

x = np.arange(
    len(labels)
)

w = 0.34

b1 = ax.bar(
    x - w / 2,
    m8,
    width=w,
    color=M8_COLOR,
    label="M8: with TEMP_C",
    zorder=3
)

b2 = ax.bar(
    x + w / 2,
    m7,
    width=w,
    color=M7_COLOR,
    label="M7: without TEMP_C",
    zorder=3
)

ax.axhline(
    0,
    color="#777777",
    lw=0.7
)

ax.set_xticks(
    x,
    labels,
    rotation=22,
    ha="right"
)

ax.set_ylabel(
    r"$R^2$"
)

all_r2 = np.r_[
    m8,
    m7
]

lo = min(
    -0.05,
    float(
        np.nanmin(
            all_r2
        )
    )
    -
    0.08
)

hi = max(
    1.0,
    float(
        np.nanmax(
            all_r2
        )
    )
    +
    0.08
)

ax.set_ylim(
    lo,
    hi
)

for bars in [
    b1,
    b2
]:
    for bar in bars:
        val = bar.get_height()

        ax.text(
            bar.get_x()
            +
            bar.get_width()
            /
            2,
            val
            +
            (
                0.025
                if val >= 0
                else -0.035
            ),
            f"{val:.2f}",
            ha="center",
            va=(
                "bottom"
                if val >= 0
                else "top"
            ),
            fontsize=7.0
        )

ax.set_title(
    "Air-temperature ablation",
    fontweight="bold",
    pad=8
)

ax.legend(
    frameon=False,
    fontsize=7.0,
    loc="lower left"
)

ax.grid(
    axis="y",
    alpha=0.16,
    zorder=0
)

ax.text(
    -0.14,
    1.02,
    "a",
    transform=ax.transAxes,
    fontsize=11,
    fontweight="bold"
)


# ------------------------------------------------------------------------------------------------
# b. USCRN R2 BY YEAR
# ------------------------------------------------------------------------------------------------

ax = axes[
    1
]

for model, color, label in [
    (
        "M8_WITH_TEMP",
        M8_COLOR,
        "M8: with TEMP_C"
    ),
    (
        "M7_NO_TEMP",
        M7_COLOR,
        "M7: without TEMP_C"
    ),
    (
        "RAW_MODIS",
        RAW_COLOR,
        "Raw MODIS LST"
    ),
]:

    q = (
        GROUND_BY_YEAR.loc[
            GROUND_BY_YEAR[
                "MODEL"
            ].eq(
                model
            )
        ]
        .sort_values(
            "YEAR"
        )
    )

    if len(q):

        ax.plot(
            q[
                "YEAR"
            ],
            q[
                "R2"
            ],
            marker="o",
            ms=4.0,
            lw=1.55,
            color=color,
            label=label
        )

ax.axhline(
    0,
    color="#777777",
    lw=0.7
)

ax.set_xlabel(
    "Year"
)

ax.set_ylabel(
    r"USCRN $R^2$"
)

ax.set_title(
    "Independent NOAA USCRN validation",
    fontweight="bold",
    pad=8
)

ax.grid(
    axis="y",
    alpha=0.16
)

ax.legend(
    frameon=False,
    fontsize=6.8,
    loc="best"
)

ax.text(
    -0.14,
    1.02,
    "b",
    transform=ax.transAxes,
    fontsize=11,
    fontweight="bold"
)


# ------------------------------------------------------------------------------------------------
# c. USCRN RMSE BY YEAR
# ------------------------------------------------------------------------------------------------

ax = axes[
    2
]

for model, color, label in [
    (
        "M8_WITH_TEMP",
        M8_COLOR,
        "M8: with TEMP_C"
    ),
    (
        "M7_NO_TEMP",
        M7_COLOR,
        "M7: without TEMP_C"
    ),
    (
        "RAW_MODIS",
        RAW_COLOR,
        "Raw MODIS LST"
    ),
]:

    q = (
        GROUND_BY_YEAR.loc[
            GROUND_BY_YEAR[
                "MODEL"
            ].eq(
                model
            )
        ]
        .sort_values(
            "YEAR"
        )
    )

    if len(q):

        ax.plot(
            q[
                "YEAR"
            ],
            q[
                "RMSE_C"
            ],
            marker="o",
            ms=4.0,
            lw=1.55,
            color=color,
            label=label
        )

ax.set_xlabel(
    "Year"
)

ax.set_ylabel(
    "USCRN RMSE (°C)"
)

ax.set_title(
    "Station-validation error",
    fontweight="bold",
    pad=8
)

ax.grid(
    axis="y",
    alpha=0.16
)

ax.text(
    -0.14,
    1.02,
    "c",
    transform=ax.transAxes,
    fontsize=11,
    fontweight="bold"
)


fig.suptitle(
    "Figure S11. Air-temperature ablation and independent NOAA USCRN validation",
    fontsize=12.0,
    fontweight="bold",
    y=0.96
)

fig.text(
    0.5,
    0.035,
    "M8 and M7 use the same training rows and the same five geographic cross-validation blocks; USCRN-nearby hexagons are excluded from model fitting.",
    ha="center",
    fontsize=7.3,
    color="#555555"
)

supp_save(
    fig,
    "Figure_S11_Air_Temperature_Ablation_USCRN_FIXED"
)

plt.show()
plt.close(
    fig
)

register(
    "S11",
    "Air-temperature ablation and independent NOAA USCRN validation"
)


In [ ]:
# ================================================================================================
# FINALIZE SUPPLEMENTARY PACKAGE
# ================================================================================================

MANIFEST_DF = pd.DataFrame(
    MANIFEST
)

MANIFEST_DF.to_csv(
    SUP_META /
    "SUPPLEMENTARY_FIGURE_MANIFEST.csv",
    index=False
)

captions = [
    "SUPPLEMENTARY FIGURE CAPTIONS",
    "=" * 80,
    "",
    "Figure S1. Complete temporal atlas of direct GVSTMR relationship states in the CONUS simulation (T1–T10).",
    "Figure S2. Complete temporal atlas of direct GVSTMR relationship states for the real NDVI–JJA LST case (2015–2024).",
    "Figure S3. Sensitivity of direct GVSTMR spatial scores, temporal scores, and nine-state classifications to spatial-neighborhood size K and Gaussian temporal bandwidth h. Values denote agreement with the manuscript baseline K=30 and h=2.",
    "Figure S4. Row-normalized transitions among the nine GVSTMR relationship states and the fraction of hexagons retaining the same state between consecutive times.",
    "Figure S5. Time-resolved predictive performance of MGWR, GTWR, and GGPR. Model-performance statistics are supporting diagnostics and are not used to define GVSTMR classes.",
    "Figure S6. Pairwise agreement among MGWR, GTWR, and GGPR under the common model–observation GVSTMR grammar, summarized by exact matrix-state agreement and Spearman agreement of spatial and temporal relationship scores.",
    "Figure S7. Full MGWR local-coefficient atlas for the environmental predictors, accompanied by predictor-specific adaptive spatial bandwidths.",
    "Figure S8. GTWR parameter sensitivity across adaptive spatial-neighbor count K and temporal scale tau, with the minimum-RMSE combination highlighted.",
    "Figure S9. Relationship between GGPR posterior predictive uncertainty and realized absolute error, together with empirical 95% predictive-interval coverage through time.",
    "Figure S10. Original-style GeoShapley summary distributions for the simulation and real cases. Horizontal position gives the GeoShapley contribution to GGPR output, point density shows the contribution distribution, and color denotes the corresponding feature value.",
    "Figure S11. Air-temperature ablation comparing the full eight-predictor model with the same model excluding air temperature, together with independent NOAA USCRN validation and annual station-validation performance.",
]

(
    SUP_META /
    "SUPPLEMENTARY_CAPTIONS.txt"
).write_text(
    "\n".join(
        captions
    ),
    encoding="utf-8"
)

with zipfile.ZipFile(
    SUP_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for p in SUP_ROOT.rglob("*"):

        if p.is_file():

            zf.write(
                p,
                arcname=str(
                    p.relative_to(
                        SUP_ROOT
                    )
                )
            )

print("\n" + "=" * 105)
print("✅ GVSTMR SUPPLEMENTARY OUTPUTS COMPLETE")
print("=" * 105)

display(
    MANIFEST_DF
)

print("\nSupplementary folder:")
print(SUP_ROOT)

print("\nFinal ZIP:")
print(SUP_ZIP)

print("\nNo file was written to Google Drive.")
